# Swiss Citation — Anchor-Funnel v7 (production diagnostic build)

**Pass criterion: R@1000 ≥ 0.60** for val_001 (≥ 26/42 gold). Stretch: ≥ 0.90 / 38+.

This notebook is structured as **11 phases** with a markdown header before every code cell.
Each markdown header states:
- **What** the next cell computes
- **Why** we need that signal (which retrieval gap it closes)
- **Expected outputs** — numeric ranges so anomalies pop out
- **Failure modes** — what to check if a number looks off

The final phase (Phase 10) does **per-gold diagnosis**: for every gold citation
NOT captured in top-1000, the notebook traces through every channel and every
signal source and prints a concrete root-cause report (with rank, score, token
overlap, cosine similarity, graph degrees).

## Phase map

| Phase | Cells | Purpose |
|---|---|---|
| 1. Setup | env, drive, paths, knobs | runtime + IO + global config |
| 2. Load val + corpus indexes | val.csv, law-llm jsonl, court-v5 jsonl | build all in-memory indexes |
| 3. Citation graph | sqlite → idx_graph_out / idx_graph_in | 4-layer graph (23.65M edges) |
| 4. Per-area bedrock + co-citation | corpus statistics | universal articles per legal area |
| 5. BM25 (FTS5 in-memory) | query-side lexical match | catches code names + Swiss terminology |
| 6. Vector channel setup | Qwen3-Embedding-8B + corpus E_GPU | semantic similarity |
| 7. Query expansion (Qwen3-32B) | LLM → structured JSON targets | concepts_en, term_targets_de/fr, statute_targets, legal_area_keywords |
| 8. Run channels | 14 retrieval signals | each with per-channel R@K |
| 9. RRF fusion + gating | reciprocal rank fusion + neg-gate | top-1000 final pool |
| 10. **Diagnosis** | per-gold trace + miss attribution | which signal failed for which gold |
| 11. Save artifacts + cleanup | persist to Drive | reproducibility |

## Constraints respected (recall from prior chat)

- **No hardcoded lists.** No DE_LEX, no fixed BGE list, no statute cluster table.
- **No query-specific knowledge.** Architecture must generalize to any val/test/production query.
- **No train data.** Train.csv is never read.
- **Open-source only.** Qwen3-32B (query expansion), Qwen3-Embedding-8B (vectors).

## v7 changes vs v6 (R@1000 was 0.357)

| Change | Why |
|---|---|
| Sibling budget 500 → 2000 | v6's 500 cap was a non-deterministic `set→list[:budget]` slice; with 2691 typical seeds × ~5 siblings, the slice dropped val_001 sibling Es. |
| Graph forward channel (NEW) | Loads corpus-derived citation graph (4 alias passes, 23.65M edges) and follows outgoing edges — surfaces text-cited targets PLUS all sibling Es of cited judgments via case-level fan-out. Closes 6/11 originally-orphan val_001 gold. |
| Graph reverse channel (NEW) | Follows incoming edges — finds rows that text-cite seeds. Co-citation expansion via the actual corpus graph, not LLM. |
| Phase 10 diagnostics | Every missed gold gets a per-channel trace + recommended fix. |

---

## v7.5 channel summary (production-grade fixes)

| # | Channel | Index used | Gap closed | Budget | RRF weight |
|---|---|---|---|---:|---:|
| 1 | law_direct_match | idx_law_direct | LLM-named statutes (incl. corpus-related codes) → matching law rows. Uncapped. | None | 1.2 |
| 2 | court_statute | idx_court_statute | LLM-named statute appears in row's `statute_anchors`. Specificity-weighted + paragraph_role boost. | 8000 | 1.5 |
| 3 | co_citation | co_neighbours + idx_law_direct + idx_court_statute | Statute cluster co-cited with LLM target (e.g., 221 StPO ↔ 212 StPO). Specificity-weighted. | 2500 | 0.7 |
| 4 | per_area_bedrock | per_area_canon_count, filtered by corpus-derived code family | Universal procedural articles per legal area (Art. 100 BGG, Art. 422 StPO etc.). top_n=1000 captures deep procedural cluster. | 1500 | 1.5 |
| 5 | statute_backprop | doc_statute_anchors | Caught court rows → law articles they cite. Rare-canon specificity weight. | 2000 | 2.5 |
| 6 | sibling_expansion | idx_court_base + idx_judgment_importance | All Es of caught judgments. Judgment-importance scoring (BGE landmark > obscure docket). | 5000 | 1.0 |
| 7 | graph_forward (v7) | idx_graph_out + idx_judgment_importance | Caught row → text-cited targets + case-level fan-out. Importance-weighted. | 5000 | 2.0 |
| 8 | graph_reverse (v7) | idx_graph_in + idx_judgment_importance | Caught row ← rows that text-cite it. Landmark filter (judgment importance ≥ 5). | 3000 | 0.5 |
| 9 | graph_2hop (v7, off) | idx_graph_out | 2-hop forward expansion. Disabled — crashed in v7.2. | 1500 | 0.0 |
| 10 | concept_en | idx_concept_en | LLM concepts → English-tagged rows. **Token-overlap matcher** (stopwords filtered, weighted by shared meaningful tokens). | 3000 | 1.8 |
| 11 | term_orig | idx_term_orig + idx_term_lemma | LLM DE/FR terms → original-language-tagged rows. **German lemmatizer** (drop -en/-e/-er/-s with 4-char floor) + substring overlap ratio. | 2500 | 1.2 |
| 12 | bm25 | FTS5 per-language (de/fr/it/en) + enhance() | **Multi-language FTS5**: 4 separate indices, query in each language with appropriate term targets, merge by max-score. | 2000 | 0.8 |
| 13 | vector_raw | E_GPU brute-force | Dense semantic match on raw query. | 2000 | 1.0 |
| 14 | vector_enriched | E_GPU brute-force | Dense semantic match on keyword-enriched query. | 2000 | 1.0 |

### Guarantee channels (7) — round-robin merged, ~130 each = 910 slots + 90 RRF tail
1. `law_direct_match`, 2. `per_area_bedrock`, 3. `statute_backprop`,
4. `concept_en` (NEW), 5. `graph_forward` (NEW), 6. `sibling_expansion` (NEW), 7. `term_orig` (NEW)

### Role-aware negative gate
Substantive paragraph roles (`facts`, `reasoning`, `legal_standard`, `application`, `holding`, `citation`, `procedural_history`) override the noisy `is_notification_paragraph` enrichment flag.


# Phase 1 — Setup

## 1.1 Environment & GPU check

**What:** Print Python version, PyTorch version, GPU name + VRAM.

**Why:** All our heavy work (Qwen3-32B, Qwen3-Embedding-8B, full-corpus dense
matmul) requires a single high-VRAM GPU. The Blackwell instance has 95.6 GB —
Qwen3-32B in bf16 (~65 GB) + Qwen3-Embedding-8B + corpus `E_GPU` (~37 GB) just
fit if loaded sequentially.

**Expected:** Python 3.12+, PyTorch 2.x with CUDA, 1× GPU with ≥ 80 GB.

**Failure modes:**
- "0 GPUs" → Colab session lost GPU; restart runtime.
- VRAM < 80 GB → Qwen3-32B will OOM. Drop to `qwen_query_model = "Qwen/Qwen3-8B"`
  in CONFIG and accept thinner JSON targets.

In [ ]:
import sys, torch
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}, {p.total_memory / 1024**3:.1f} GB")
else:
    print("[WARN] No CUDA GPU available — vector channels will be skipped.")

Python: 3.12.13
PyTorch: 2.10.0+cu128
  GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition, 95.0 GB


## 1.2 Mount Google Drive

**What:** Mount Drive so we can read the corpus, embeddings, and citation
graph from `/content/drive/MyDrive/swiss_law/`.

**Why:** The 21 GB of fp16 embeddings, 24 GB unified retrieval SQLite, and
2.4 GB citation graph all live on Drive — too large to download per run.

**Expected:** "Mounted at /content/drive". Skip silently if running locally.

**Failure modes:**
- "Drive not authorized" → click the OAuth link Colab prints.
- Mount succeeds but `MyDrive/swiss_law` is empty → wrong account; remount.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"[skip] Not on Colab or drive already mounted: {e}")

Mounted at /content/drive


## 1.3 Resolve all data paths

**What:** Auto-detect `DATA_ROOT` (could be Drive, local, or Kaggle) and
build a `PATHS` dict pointing at every required artifact.

**Why:** Same notebook should run on Colab, Kaggle, or locally without code
changes. Each artifact is checked for existence with a clear OK/MISSING flag.

**Required inputs:**
- `data/val.csv` — 10 English queries with gold citations.
- `law_llm_descriptors_*.jsonl` — LLM enrichment of all 175k laws.
- `court_authority_cards_v5_unified.jsonl` — court enrichment (10.4 GB).
- `embeddings/qwen3_8b_unified_chunk*.npy` — 27 fp16 chunks (21 GB total).
- `embeddings/qwen3_8b_unified_manifest.parquet` — doc_id ↔ row_index.
- **v7 NEW:** `data_insights/citation_graph_extracted.sqlite` — 4-layer graph (~2.4 GB).

**Expected:** All flags `OK`. If `graph_db` is `MISSING`, graph channels
silently skip (recall drops back to v6 levels).

**Failure modes:**
- `MISSING law_llm` or `court_v5` → notebook can't build any index. Stop.
- `MISSING emb_dir` → vector channels skipped; BM25 + anchors still run.
- `MISSING graph_db` → graph channels skipped; sibling_expansion (court_base) only.

In [ ]:
from pathlib import Path

CANDIDATE_ROOTS = [
    Path("/content/drive/MyDrive/swiss_law"),
    Path("/content/drive/MyDrive/swiss_citation_extraction"),
    Path("/content/swiss_citation_extraction"),
    Path(r"E:/swiss_citation_extraction"),
    Path.cwd(),
]

DATA_ROOT = None
for root in CANDIDATE_ROOTS:
    if (root / "data" / "val.csv").exists():
        DATA_ROOT = root; break
if DATA_ROOT is None:
    print("[warn] Could not auto-detect DATA_ROOT — defaulting to /content/drive/MyDrive/swiss_law")
    DATA_ROOT = Path("/content/drive/MyDrive/swiss_law")
print(f"DATA_ROOT = {DATA_ROOT}")

def first_existing(*paths):
    for p in paths:
        if p.exists(): return p
    return paths[0]

PATHS = {
    "val_csv": DATA_ROOT / "data" / "val.csv",
    "law_llm": first_existing(
        DATA_ROOT / "data" / "checkpoints" / "law_llm_descriptors_0000000_all.jsonl",
        DATA_ROOT / "law_json_llm_output" / "law_llm_descriptors_0000000_all.jsonl",
    ),
    "court_v5": first_existing(
        DATA_ROOT / "artifacts_v2" / "court_authority_cards_v5_unified.jsonl",
        DATA_ROOT / "artifacts" / "court_authority_cards_v5_unified.jsonl",
    ),
    "emb_dir":      DATA_ROOT / "artifacts" / "embeddings",
    "emb_manifest": DATA_ROOT / "artifacts" / "embeddings" / "qwen3_8b_unified_manifest.parquet",
    "graph_db": first_existing(
        DATA_ROOT / "data_insights" / "citation_graph_extracted.sqlite",
        DATA_ROOT / "citation_graph_extracted.sqlite",
    ),
    "out_dir":      DATA_ROOT / "research" / "anchor_funnel_val001_v7",
}
PATHS["out_dir"].mkdir(parents=True, exist_ok=True)

for k, p in PATHS.items():
    if k == "out_dir": continue
    flag = "OK     " if p.exists() else "MISSING"
    print(f"  {flag}  {k:<14} {p}")

EMB_AVAILABLE   = PATHS["emb_dir"].exists() and any(PATHS["emb_dir"].glob("qwen3_8b_unified_chunk*.npy"))
GRAPH_AVAILABLE = PATHS["graph_db"].exists()
print()
print(f"  EMB_AVAILABLE   = {EMB_AVAILABLE}")
print(f"  GRAPH_AVAILABLE = {GRAPH_AVAILABLE}")

DATA_ROOT = /content/drive/MyDrive/swiss_law
  OK       val_csv        /content/drive/MyDrive/swiss_law/data/val.csv
  OK       law_llm        /content/drive/MyDrive/swiss_law/data/checkpoints/law_llm_descriptors_0000000_all.jsonl
  OK       court_v5       /content/drive/MyDrive/swiss_law/artifacts_v2/court_authority_cards_v5_unified.jsonl
  OK       emb_dir        /content/drive/MyDrive/swiss_law/artifacts/embeddings
  OK       emb_manifest   /content/drive/MyDrive/swiss_law/artifacts/embeddings/qwen3_8b_unified_manifest.parquet
  OK       graph_db       /content/drive/MyDrive/swiss_law/data_insights/citation_graph_extracted.sqlite

  EMB_AVAILABLE   = True
  GRAPH_AVAILABLE = True


## 1.4 Knob panel (CONFIG)

**What:** All architecture knobs live in a single `CONFIG` dict — budgets,
RRF k, BM25 limits, vector top-k, query-expansion model, etc.

**Why:** Editing budgets without touching channel code is essential when
diagnosis suggests "channel X dropped this gold due to truncation, lift
budget". Every budget is rationalized in a comment.

**Sensitive knobs:**
- `budget_sibling = 2000` — was 500 in v6, dropped sibling Es non-deterministically.
- `budget_graph_forward = 1500` — graph 1-hop forward; case-level fan-out adds
  up to ~30 expansions per seed, 1500 covers 50 seeds × 30 siblings.
- `budget_graph_reverse = 1000` — co-citing rows; bounded to keep ranking signal.
- `enable_graph_2hop = False` — 2-hop tends to dump procedural articles already
  caught elsewhere; flip to True only after Phase 10 says so.

**Expected:** dict prints cleanly; nothing should be `None` except `budget_law_direct`.

In [ ]:
import json

CONFIG = {
    "topk_final": 1000,

    # --- Channel budgets ---------------------------------------------------
    # law_direct_match: every law row whose canonical citation matches a
    # (LLM-named OR co-cited) statute target. Tiny per canon; uncapped is safe.
    "budget_law_direct":     None,

    # court_statute: court rows annotated with one of the LLM-named statutes.
    "budget_court_statute":  8000,   # v7.5: lifted 600->8000 (multi-match scoring + role boost now have room)

    # concept_en: rows whose concepts_en token overlaps with LLM concept_targets.
    "budget_concept":        3000,   # v7.5: lifted (rank-200+ gold lost otherwise)

    # term_orig: rows whose terms_original (DE/FR/IT) overlap with LLM term_targets.
    "budget_term":           2500,   # v7.5: lifted (deep-rank gold survives)

    # per_area_bedrock: most-cited canonical statutes within the LLM-named legal_area.
    # Per-area_top_n is internal cap; this budget caps the bedrock channel output.
    "budget_per_area":       1500,   # v7.5: procedural cluster lives at rank 100-500

    # co_citation: for each LLM statute target, fetch top-K co-citation neighbours
    # and pull their law + court rows.
    "budget_co_citation":    2500,   # v7.5: more co-cited candidates

    # bm25: lexical match on enriched query text. enhance() adds top-K corpus-
    # associated codes to query.
    "budget_bm25":           2000,   # v7.5: multi-language each gets 500

    # vector_raw / vector_enriched: dense semantic match using Qwen3-Embedding-8B
    # against full-corpus E_GPU.
    "budget_vector":         2000,   # v7.5: Obs 3 ceiling 0.289 — need deeper pool
    "budget_vector_enriched":2000,

    # statute_backprop: each caught court row contributes its cited statutes;
    # score = number of distinct caught court rows citing that article.
    # Surfaces procedural cluster (Art. 100 BGG, Art. 422 StPO, etc.).
    "budget_backprop":       2000,   # v7.5: rare-specificity scoring + more law candidates

    # sibling_expansion (court_base): caught court row → all Es of same judgment.
    # v6 had 500 → non-deterministic slice dropped val_001 sibling Es. 2000 fits
    # 100 seeds × 20 Es each.
    "budget_sibling":        5000,   # v7.5: judgment-importance scoring + bigger pool

    # graph_forward / graph_reverse / graph_2hop: 1-hop and 2-hop traversal of
    # the citation graph (intra-judgment backrefs + date aliases + range +
    # case-level fan-out). budget_forward sized for case-level fan-out from
    # ~50 seeds × 30 sibling-fanout ≈ 1500.
    "budget_graph_forward":  5000,   # v7.5: case-level fan-out + judgment importance
    "budget_graph_reverse":  3000,
    "enable_graph_2hop":     False,
    "budget_graph_2hop":     1500,

    # --- RRF + guarantee ---------------------------------------------------
    "rrf_k": 60,
    # Channels whose hits are PREPENDED before the RRF tail in Phase 9.
    # Keep this list small — graph_forward/reverse compete via RRF, only the
    # high-precision channels are guaranteed.
    # v7.5: 7-channel guarantee with smaller per-channel cap. Round-robin
    # ensures each high-recall channel contributes regardless of RRF score.
    # 7 channels × cap=130 = 910 guarantee slots, leaves ~90 RRF tail.
    "guarantee_channels": [
        "law_direct_match",
        "per_area_bedrock",
        "statute_backprop",
        "concept_en",            # v7.5 NEW
        "graph_forward",         # v7.5 NEW
        "sibling_expansion",     # v7.5 NEW
        "term_orig",             # v7.5 NEW
    ],
    "guarantee_per_channel": 130,
    # v7.5 weights — updated based on v7.4 per-channel mean recall + structural
    # value (multi-channel role).
    "channel_weights": {
        "statute_backprop":  2.5,   # mean recall 0.453 — universal best
        "graph_forward":     2.0,   # mean recall 0.324 — case-level fan-out
        "concept_en":        1.8,   # v7.5 lifted — token-overlap matcher now strong
        "court_statute":     1.5,   # v7.5 lifted — specificity+role scoring
        "per_area_bedrock":  1.5,
        "vector_raw":        1.0,
        "vector_enriched":   1.0,
        "term_orig":         1.2,   # lemma+substring scoring
        "law_direct_match":  1.2,
        "sibling_expansion": 1.0,
        "bm25":              0.8,
        "co_citation":       0.7,
        "graph_reverse":     0.5,
        "graph_2hop":        0.0,
    },
    # v7.5 NEW: corpus-derived code-family expansion for per_area_bedrock.
    # When LLM names "StPO", we ALSO admit StBOG/BGG/BV/EMRK canons via
    # corpus co-citation evidence (not a hardcoded list).
    "code_family_top_k": 8,

    # --- Per-area bedrock --------------------------------------------------
    "per_area_top_n": 1000,   # v7.5: procedural cluster (422/428/135 StPO) lives at rank 100-500

    # --- Co-citation -------------------------------------------------------
    "co_citation_top_k_per_target":    50,   # v7.5: more cluster expansion
    "co_citation_min_co_count":        50,
    # drop neighbours that are TOO globally common (Art. 36 BV cited everywhere
    # would pollute court_statute). 5000 = ~0.2% of 2.5M corpus.
    "co_citation_max_neighbour_count": 50000,  # v7.5: allow more common; specificity downranks noise

    # --- Concept matching --------------------------------------------------
    "concept_substring_top_k": 6,

    # --- BM25 --------------------------------------------------------------
    "bm25_max_query_terms": 60,    # cap to stop token-explosion from enriched query
    "bm25_min_token_len":   3,

    # --- Vector ------------------------------------------------------------
    "vector_emb_model": "Qwen/Qwen3-Embedding-8B",
    "vector_topk":      800,

    # --- Query expansion ---------------------------------------------------
    "qwen_query_model":    "Qwen/Qwen3-32B",
    "qwen_max_new_tokens": 1024,

    # --- enhance() — corpus-derived BM25 lexicon expansion -----------------
    "enhance_top_k_codes":   5,
    "enhance_repeat_count":  5,
    "enhance_min_idf":       1.0,

    # --- Negative gate -----------------------------------------------------
    "noise_paragraph_roles": {"notification", "header", "empty", "metadata"},

    "lowercase_concepts": True,
    "lowercase_terms":    True,
}

print(json.dumps({k: v for k, v in CONFIG.items() if not isinstance(v, set)}, indent=2, default=str))

{
  "topk_final": 1000,
  "budget_law_direct": null,
  "budget_court_statute": 600,
  "budget_concept": 600,
  "budget_term": 500,
  "budget_per_area": 150,
  "budget_co_citation": 1000,
  "budget_bm25": 600,
  "budget_vector": 800,
  "budget_vector_enriched": 800,
  "budget_backprop": 800,
  "budget_sibling": 2000,
  "budget_graph_forward": 1500,
  "budget_graph_reverse": 1000,
  "enable_graph_2hop": false,
  "budget_graph_2hop": 1000,
  "rrf_k": 60,
  "guarantee_channels": [
    "law_direct_match",
    "per_area_bedrock",
    "statute_backprop"
  ],
  "guarantee_per_channel": 1000,
  "channel_weights": {
    "statute_backprop": 2.5,
    "graph_forward": 2.0,
    "court_statute": 0.7,
    "bm25": 0.7,
    "term_orig": 0.7,
    "sibling_expansion": 0.7,
    "graph_reverse": 0.5,
    "co_citation": 0.3,
    "graph_2hop": 0.0
  },
  "per_area_top_n": 100,
  "co_citation_top_k_per_target": 20,
  "co_citation_min_co_count": 50,
  "co_citation_max_neighbour_count": 15000,
  "concept_substri

## 1.5 Load val_001 query + gold

**What:** Read `val.csv`, pick query_id=`val_001`, parse semicolon-separated
gold citations.

**Why:** val_001 is the canary query. Its gold (42 citations) spans laws
(StPO, BGG, StGB, StBOG) AND court (BGE 137 IV 122, BGE 132 I 21, multiple 1B/7B dockets).

**Expected:** 42 gold citations. Query starts "May a court lawfully order
a three-month extension of pre-trial detention...".

**Failure modes:**
- "0 gold" → check the `gold_citations` column name in val.csv hasn't drifted.
- Wrong query_id → confirm row matches.

In [ ]:
import pandas as pd

val_df = pd.read_csv(PATHS["val_csv"])
print(f"val.csv has {len(val_df)} queries")
row = val_df[val_df["query_id"] == "val_001"].iloc[0]
QUERY = str(row["query"])
val_gold = [c.strip() for c in str(row["gold_citations"]).split(";") if c.strip()]
print(f"val_001: {len(val_gold)} gold citations")
print(f"Query (first 240 chars): {QUERY[:240]}{'...' if len(QUERY) > 240 else ''}")
print()
print("First 6 gold citations:")
for g in val_gold[:6]:
    print(f"  - {g}")

val.csv has 10 queries
val_001: 42 gold citations
Query (first 240 chars): May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 Abs. 1 lit. b StPO (risk of collusion) consistent with the principle of proportionality when the accused—detained after an alleged late‑night assault a...

First 6 gold citations:
  - Art. 221 Abs. 1 StPO
  - Art. 140 Abs. 1 StGB
  - Art. 396 Abs. 1 StPO
  - Art. 222 StPO
  - Art. 393 Abs. 1 StPO
  - Art. 382 Abs. 1 StPO


# Phase 2 — Build all corpus indexes (one pass)

## 2.1 What this big cell does

This cell is the heart of the retrieval pipeline. It **streams both JSONL files**
(`law_llm_descriptors` + `court_authority_cards_v5`) and builds **all** in-memory
indexes the channels need:

| Index | Type | What it maps |
|---|---|---|
| `cit_to_doc_ids[citation]` | dict[str, list[str]] | citation string → list of doc_ids |
| `doc_meta[did]` | dict[str, dict] | doc_id → {citation, family, court_base, paragraph_role} |
| `idx_law_direct[canonical]` | dict[str, set] | "100 BGG" → law doc_ids |
| `idx_court_statute[canonical]` | dict[str, set] | "100 BGG" → court rows whose anchors include this |
| `idx_court_base[base]` | dict[str, set] | "137 IV 122" → all Es of judgment (used by sibling_expansion) |
| `idx_concept_en[token]` | dict[str, set] | English concept → doc_ids |
| `idx_term_orig[token]` | dict[str, set] | DE/FR/IT term → doc_ids |
| `search_text[did]` | dict[str, str] | doc_id → BM25-search text (concatenated enrichment) |
| `legal_area_per_doc[did]` | dict[str, str] | court doc → its `legal_area_static` |
| `co_citation_pairs` | Counter[(canon_a, canon_b)] | unordered pairs of statutes co-cited within same court row |
| `tlf[token][code]` | dict[str, Counter] | corpus-wide token → law-code association (for `enhance()`) |
| `doc_statute_anchors[did]` | dict[str, set] | court doc_id → set of canonical statutes it cites |

## 2.2 Why we need each one (channel attribution)

- `idx_law_direct` → channels: `law_direct_match`, `per_area_bedrock`, `statute_backprop`, `co_citation`
- `idx_court_statute` → channels: `court_statute`, `co_citation`
- `idx_court_base` → channel: `sibling_expansion` (own-judgment Es)
- `idx_concept_en` / `idx_term_orig` → channels: `concept_en`, `term_orig`
- `search_text` → channel: `bm25` (FTS5 indexed in Phase 5)
- `co_citation_pairs` → Phase 4 builds `co_neighbours` from this; channel: `co_citation`
- `tlf` → BM25 query enhancement in Phase 5
- `doc_statute_anchors` → channel: `statute_backprop`

## 2.3 Expected outputs
- Law: ~175k rows in ~10 s
- Court: ~2.47M rows in ~150 s
- Token→code association: ~90k tokens, avg ~10 codes/token
- Co-citation pairs: ~1.1M

## 2.4 Failure modes
- **Slower than 200 s for court** → Drive throttling; retry.
- **`Total docs ≠ ~2.65M`** → JSONL truncation; check file size matches local.
- **`tlf` very small (< 50k tokens)** → law tokenizer pattern wrong; check the
  regex split below.

In [ ]:
from collections import defaultdict, Counter
import re, json, time

# --- Statute / case canonicalizers --------------------------------------------
CODE_ALIAS = {
    "CPP": "StPO", "CP": "StGB", "CC": "ZGB", "CO": "OR",
    "LTF": "BGG", "LACI": "AVIG", "LAA": "UVG",
    "LP": "SchKG", "LDIP": "IPRG", "Cst": "BV", "Cst.": "BV",
    "STPO": "StPO", "OBG": "OR",
}
ART_RE = re.compile(r"art\.?\s*(\d+[a-z]?)", re.I)
CODE_RE = re.compile(r"\b([A-Z][A-Za-z]{1,8}\.?)\b")

def statute_anchor_canonical(raw):
    if not raw: return None
    s = raw.strip()
    m = ART_RE.search(s)
    if not m: return None
    cands = [c.strip(".") for c in CODE_RE.findall(s)
             if c.strip(".") not in ("Art","Abs","Ziff","lit","let","al","Bst")]
    if not cands: return None
    code = CODE_ALIAS.get(cands[-1], cands[-1])
    return f"{m.group(1)} {code}"

def article_num(raw):
    if not raw: return None
    m = ART_RE.search(raw.strip())
    return m.group(1) if m else None

LEGAL_AREA_DEFAULT_CODE = {
    "criminal law and criminal procedure": "StPO",
    "criminal procedure":                  "StPO",
    "criminal law":                        "StGB",
    "civil law":                           "ZGB",
    "obligations":                         "OR",
    "civil procedure":                     "ZPO",
    "constitutional and public law":       "BV",
    "constitutional law":                  "BV",
    "administrative law":                  "VwVG",
    "social insurance":                    "ATSG",
    "tax law":                             "DBG",
}

def canonicalize_row_anchors(raw_anchors, legal_area_static):
    canons = set()
    primary_code = None
    for sa in raw_anchors:
        c = statute_anchor_canonical(sa)
        if c:
            primary_code = c.split()[1]; break
    fallback = primary_code
    if fallback is None and legal_area_static:
        la = legal_area_static.lower()
        for k, v in LEGAL_AREA_DEFAULT_CODE.items():
            if k in la:
                fallback = v; break
    for sa in raw_anchors:
        c = statute_anchor_canonical(sa)
        if c:
            canons.add(c); continue
        n = article_num(sa)
        if n and fallback:
            canons.add(f"{n} {fallback}")
    return canons

CASE_BGE_RE    = re.compile(r"BGE\s+(\d+)\s+([IVX]+)\s+(\d+)")
CASE_DOCKET_RE = re.compile(r"\b(\d[A-Z]_\d+/\d{4})\b")

def case_anchor_canonical(raw):
    if not raw: return None
    s = raw.strip()
    m = CASE_BGE_RE.search(s)
    if m: return f"BGE {m.group(1)} {m.group(2)} {m.group(3)}"
    m = CASE_DOCKET_RE.search(s)
    if m: return m.group(1)
    return None

TOKEN_NORM_RE = re.compile(r"\s+")
def norm_token(s, lower):
    if not s: return None
    s = TOKEN_NORM_RE.sub(" ", s.strip())
    if not s: return None
    return s.lower() if lower else s

# --- German lemmatizer for term_orig channel ---------------------------------
# Drop one common inflectional suffix at a time. Only drop a suffix if the
# remaining stem is >= 4 chars; allow recursion (e.g. 'haftens' -> 'haften'
# -> 'haft'). Surface form is always kept in idx_term_orig as well, so this
# is purely additive.
_TERM_LEMMA_SUFFIXES = ("en", "es", "em", "er", "e", "n", "s")
def term_lemma(tok):
    if not tok: return tok
    cur = tok
    seen = {cur}
    while True:
        changed = False
        for suf in _TERM_LEMMA_SUFFIXES:
            if cur.endswith(suf) and len(cur) - len(suf) >= 4:
                stem = cur[: len(cur) - len(suf)]
                if stem not in seen:
                    cur = stem; seen.add(cur); changed = True; break
        if not changed:
            break
    return cur

# --- Indexes -----------------------------------------------------------------
cit_to_doc_ids       = defaultdict(list)
doc_meta             = {}
idx_law_direct       = defaultdict(set)
idx_court_statute    = defaultdict(set)
idx_case_anchor      = defaultdict(set)
idx_court_base       = defaultdict(set)
idx_concept_en       = defaultdict(set)
idx_term_orig        = defaultdict(set)
idx_term_lemma       = defaultdict(set)   # lemma-form -> set(doc_ids)
term_orig_keys       = set()              # all normalized surface-form keys (for substring scan)
legal_area_per_doc   = {}
search_text          = {}
co_citation_pairs    = Counter()
tlf                  = defaultdict(Counter)
token_doc_count      = Counter()
doc_statute_anchors  = {}
doc_language         = {}

DOC_ID_LAW   = lambda i: f"law:{i}"
DOC_ID_COURT = lambda i: f"court:{i}"

def _take_text(*parts, max_chars=2000):
    out = []
    for p in parts:
        if not p: continue
        if isinstance(p, list):
            for x in p:
                if isinstance(x, str): out.append(x)
                elif isinstance(x, dict):
                    for v in x.values():
                        if isinstance(v, str): out.append(v)
        elif isinstance(p, str):
            out.append(p)
    return (" ".join(out))[:max_chars]

# --- Stream law jsonl --------------------------------------------------------
t0 = time.time(); n_law = 0
with open(PATHS["law_llm"], encoding="utf-8") as f:
    for line in f:
        try: obj = json.loads(line)
        except Exception: continue
        cit = obj.get("citation","")
        if not cit: continue
        did = DOC_ID_LAW(n_law)
        cit_to_doc_ids[cit].append(did)
        doc_meta[did] = {"citation": cit, "family": "law", "court_base": None,
                         "paragraph_role": None, "is_notification_paragraph": False}
        canon = statute_anchor_canonical(cit)
        if canon: idx_law_direct[canon].add(did)

        enr = obj.get("llm_enrichment") or {}
        terms_de = []; terms_en = []
        for t in enr.get("terms_de_to_en") or []:
            if isinstance(t, dict):
                de = norm_token(t.get("de",""), CONFIG["lowercase_terms"])
                en = norm_token(t.get("en",""), CONFIG["lowercase_terms"])
                if de:
                    idx_term_orig[de].add(did); terms_de.append(de)
                    term_orig_keys.add(de)
                    _lem_de = term_lemma(de)
                    if _lem_de and _lem_de != de: idx_term_lemma[_lem_de].add(did)
                    idx_term_lemma[de].add(did)
                if en:
                    idx_concept_en[en].add(did); terms_en.append(en)
        for c in enr.get("concepts_en") or []:
            tok = norm_token(c, CONFIG["lowercase_concepts"])
            if tok: idx_concept_en[tok].add(did)
        search_text[did] = _take_text(
            cit, enr.get("english_summary",""), enr.get("legal_rule",""),
            enr.get("legal_question",""), enr.get("applicability_conditions"),
            enr.get("concepts_en"), terms_de, terms_en,
        )
        legal_area_per_doc[did] = "law"
        doc_language[did] = (obj.get("language") or "de").lower()

        # token -> code association (only law rows have a clean canonical code).
        if canon and " " in canon:
            row_code = canon.split()[1].lower()
            row_text = search_text[did].lower()
            row_tokens = set()
            for tok in re.split(r"[^\w\d]+", row_text, flags=re.UNICODE):
                if len(tok) >= 3:
                    row_tokens.add(tok)
            for tok in row_tokens:
                tlf[tok][row_code] += 1
                token_doc_count[tok] += 1

        n_law += 1

print(f"Law: {n_law:,} rows indexed in {time.time()-t0:.1f}s")
print(f"Token->code association: {len(tlf):,} tokens, "
      f"avg codes/token = {sum(len(c) for c in tlf.values())/max(1,len(tlf)):.1f}")

# --- Stream court jsonl ------------------------------------------------------
t1 = time.time(); n_court = 0
with open(PATHS["court_v5"], encoding="utf-8") as f:
    for line in f:
        try: obj = json.loads(line)
        except Exception: continue
        cit = obj.get("citation","")
        if not cit: continue
        did = DOC_ID_COURT(n_court)
        cit_to_doc_ids[cit].append(did)
        cb  = obj.get("court_base") or ""
        rag = obj.get("rag_enrichment") or {}
        legal_area_static = obj.get("legal_area_static") or rag.get("legal_area") or ""
        doc_meta[did] = {"citation": cit, "family": "court", "court_base": cb,
                         "paragraph_role": rag.get("paragraph_role"),
                         "is_notification_paragraph": bool(obj.get("is_notification_paragraph"))}
        legal_area_per_doc[did] = (legal_area_static or "").lower()
        _row_lang_raw = (obj.get("language") or "").lower()
        doc_language[did] = _row_lang_raw if _row_lang_raw in ("de", "fr", "it", "en") else "en"

        if cb:
            idx_court_base[cb].add(did)
            cb_canon = case_anchor_canonical(cb)
            if cb_canon: idx_case_anchor[cb_canon].add(did)

        row_canons = canonicalize_row_anchors(rag.get("statute_anchors") or [], legal_area_static)
        for canon in row_canons:
            idx_court_statute[canon].add(did)
        if row_canons:
            doc_statute_anchors[did] = row_canons
        rc = sorted(row_canons)
        for i in range(len(rc)):
            for j in range(i+1, len(rc)):
                co_citation_pairs[(rc[i], rc[j])] += 1

        for ca in rag.get("case_anchors") or []:
            canon = case_anchor_canonical(ca)
            if canon: idx_case_anchor[canon].add(did)
        for c in rag.get("concepts_en") or []:
            tok = norm_token(c, CONFIG["lowercase_concepts"])
            if tok: idx_concept_en[tok].add(did)
        for t in rag.get("terms_original") or []:
            tok = norm_token(t, CONFIG["lowercase_terms"])
            if tok:
                idx_term_orig[tok].add(did)
                term_orig_keys.add(tok)
                _lem = term_lemma(tok)
                if _lem and _lem != tok: idx_term_lemma[_lem].add(did)
                idx_term_lemma[tok].add(did)

        search_text[did] = _take_text(
            cit, obj.get("text_excerpt_original",""),
            rag.get("concepts_en"), rag.get("terms_original"),
            rag.get("micro_topic",""), rag.get("topic",""), rag.get("subtopic",""),
            rag.get("statute_anchors"),
        )

        n_court += 1
        if n_court % 500_000 == 0:
            print(f"  court progress: {n_court:,} rows ({time.time()-t1:.1f}s)")

print(f"Court: {n_court:,} rows indexed in {time.time()-t1:.1f}s")
# Pre-compute per-canon document counts for specificity weighting in channel_court_statute.
idx_court_statute_count = {canon: len(s) for canon, s in idx_court_statute.items()}
print(f"Total docs:        {len(doc_meta):,}")
print(f"Unique citations:  {len(cit_to_doc_ids):,}")
print(f"Index sizes:       law_direct={len(idx_law_direct):,}, court_statute={len(idx_court_statute):,}, "
      f"case={len(idx_case_anchor):,}, court_base={len(idx_court_base):,}, "
      f"concept={len(idx_concept_en):,}, term={len(idx_term_orig):,}, "
      f"term_lemma={len(idx_term_lemma):,}, term_keys={len(term_orig_keys):,}")
print(f"Co-citation pairs: {len(co_citation_pairs):,}")

Law: 173,033 rows indexed in 11.2s
Token->code association: 91,173 tokens, avg codes/token = 11.7
  court progress: 500,000 rows (31.0s)
  court progress: 1,000,000 rows (65.3s)
  court progress: 1,500,000 rows (100.0s)
  court progress: 2,000,000 rows (138.2s)
Court: 2,476,315 rows indexed in 166.2s
Total docs:        2,649,348
Unique citations:  2,158,211
Index sizes:       law_direct=49,288, court_statute=81,988, case=157,227, court_base=178,593, concept=292,946, term=363,331, term_lemma=521,146, term_keys=363,331
Co-citation pairs: 1,142,790


## 2.5 Map gold to doc_ids (sanity check)

**What:** For each val_001 gold citation, look up its doc_ids in `cit_to_doc_ids`.

**Why:** If a gold citation has NO doc_id in our corpus, no channel can ever
catch it — it's a Class C miss (gold not in corpus at all). val should be
100% Class A, so all 42 gold should map. Any drop here is a corpus-build
problem, not retrieval.

**Expected:** "Mapped gold: 42/42, Total gold doc_ids: 42".

**Failure modes:**
- < 42 mapped → corpus build is missing those rows (granularity mismatch?
  maybe the gold uses paragraph-level form like "Art. 100 Abs. 1 BGG" but
  corpus has only "Art. 100 BGG"). Investigate before continuing.

In [ ]:
gold_doc_set = set()
unmapped_gold = []
for g in val_gold:
    g = g.strip()
    if not g: continue
    dids = cit_to_doc_ids.get(g, [])
    if not dids:
        unmapped_gold.append(g)
    else:
        gold_doc_set.update(dids)

total_gold = len(val_gold)
print(f"Mapped gold: {total_gold - len(unmapped_gold)}/{total_gold}")
print(f"Total gold doc_ids: {len(gold_doc_set)}")
if unmapped_gold:
    print(f"\n[WARN] {len(unmapped_gold)} gold citations have NO matching doc_id in corpus:")
    for g in unmapped_gold:
        print(f"  - {g}")
    print("These cannot be retrieved by any channel. Investigate corpus build before continuing.")

Mapped gold: 42/42
Total gold doc_ids: 42


# Phase 3 — Citation graph (v7 NEW)

## 3.1 What this cell does

Loads `data_insights/citation_graph_extracted.sqlite` (built locally via 4
alias passes; ~2.4 GB; 23.65 M edges) and converts edge tuples (citation_str
→ citation_str) into doc_id-keyed in-memory dicts:
- `idx_graph_out[did]` → list of doc_ids the row text-cites or fans-out to
- `idx_graph_in[did]` → list of doc_ids whose text cites this row

## 3.2 Why we need this — the 4 layers

The graph encodes signal that's **invisible to BM25, vector, and concepts**:

1. **Intra-judgment back-references**
   - Court text uses bare back-refs like `(vgl. E. 6.2 hiervor)` — references
     to siblings of the same judgment. Original `extract_citation_graph.py`
     missed all of these (its CONSIDERATION_RE only fires after a docket).
   - Pass 1 (`extract_intra_judgment_backrefs.py`) handles 4 patterns × 3
     languages: `vgl./siehe E. N`, `E. N hiervor`, `E. N ci-dessus`, `cf. supra
     consid. N`, including `siehe oben E. N`, `hiervor E. N`, `vorstehend`.
   - Self-tested with 15 real-world cases; 4-layer audit on 500 marker-rows
     dropped zero-target rate from 335→175 (90% of remaining are TRUE false
     markers like `nach oben` = "to the top").

2. **Date-stripped aliases**
   - Corpus extraction stores dated form `1B_210/2023 12.05.2023 E. 3`, but
     val gold uses un-dated form `1B_210/2023 E. 3`.
   - Pass 2 adds alias edges from dated → un-dated forms (only when the
     un-dated form exists as a real corpus row).

3. **E.-range expansion**
   - Corpus stores ranges as one citation: `1B_90/2021 E. 2.1-2.4`. Gold uses
     individual Es: `E. 2.1`, `E. 2.2`, `E. 2.3`, `E. 2.4`.
   - Pass 3 enumerates ranges and adds aliases.

4. **Case-level fan-out**
   - When a source cites one E. of a judgment, gold may include OTHER Es of
     the same judgment that nobody text-cites by exact pinpoint. Treats
     "citing one E." as "this case is relevant".
   - Pass 4 fans out: edge → BASE E. X spawns alias edges to all `BASE E. Y`
     where Y exists as a real corpus row.
   - Largest pass: +18.77 M edges.

After all 4 passes: **0/42 val_001 gold orphan** (was 11 originally).

## 3.3 Why doc_id mapping skips synthetic targets

Some graph nodes are synthetic (e.g., a backref `E. 4 hiervor` resolves to
`{base} E. 4` which may not be a real corpus row). Such targets have no
`cit_to_doc_ids` entry and are skipped here — graph channels only retrieve
real corpus rows.

## 3.4 Expected outputs
- ~24 M edges loaded (some skipped because synthetic targets have no doc_id)
- Out-degree avg ~10–15 (reflects case-level fan-out per cited judgment)
- In-degree avg ~10–15
- Load time: 30–60 s

## 3.5 Failure modes
- `graph_db MISSING` → graph channels skipped; sibling_expansion still works.
- `0 edges loaded` → all citations failed to map; check `cit_to_doc_ids`
  was built before this cell, and citations strings are exact (case, spacing).
- `Out-degree avg < 3` → mapping mostly failed; check date format normalization.

In [ ]:
import sqlite3 as _sqlite3

idx_graph_out = defaultdict(list)
idx_graph_in  = defaultdict(list)
GRAPH_OK = bool(GRAPH_AVAILABLE)

if GRAPH_OK:
    _t = time.time()
    _cit_to_did = {cit: dids[0] for cit, dids in cit_to_doc_ids.items() if dids}
    print(f"Built citation->doc_id map ({len(_cit_to_did):,} entries)")

    _g = _sqlite3.connect(str(PATHS["graph_db"]))
    n_loaded = 0; n_skipped = 0
    for _src, _tgt in _g.execute(
        "SELECT source, target FROM edges WHERE dataset='court_considerations'"
    ):
        _s = _cit_to_did.get(_src)
        _t2 = _cit_to_did.get(_tgt)
        if _s is None or _t2 is None:
            n_skipped += 1; continue
        if _s == _t2: continue
        idx_graph_out[_s].append(_t2)
        idx_graph_in[_t2].append(_s)
        n_loaded += 1
    _g.close()
    print(f"Graph: {n_loaded:,} edges loaded, {n_skipped:,} skipped (cit not in corpus)")
    print(f"Graph: out-degree avg = {n_loaded/max(1,len(idx_graph_out)):.1f}, "
          f"in-degree avg = {n_loaded/max(1,len(idx_graph_in)):.1f}")
    print(f"Graph: load time {time.time()-_t:.1f}s")
else:
    print("[skip] Graph DB missing — graph channels will return empty lists.")

# --- Judgment-importance index (v7.4) -------------------------------------
# importance(court_base) = sum over did in idx_court_base[court_base] of
#                         len(idx_graph_in[did])
# i.e. total incoming citations across every E.-paragraph row of the
# judgment. Landmark BGE cases score thousands; obscure dockets score 0-50.
# Used by sibling_expansion / graph_forward / graph_reverse channels in
# cell 32 to break score=1 ties and prioritize widely-cited judgments.
idx_judgment_importance = {}
if GRAPH_OK:
    _ti = time.time()
    for _cb, _dids in idx_court_base.items():
        _s = 0
        for _d in _dids:
            _s += len(idx_graph_in.get(_d, ()))
        idx_judgment_importance[_cb] = _s
    _imp_vals = list(idx_judgment_importance.values())
    if _imp_vals:
        _imp_vals_sorted = sorted(_imp_vals)
        _n = len(_imp_vals_sorted)
        print(f"Judgment importance: {len(idx_judgment_importance):,} judgments, "
              f"median={_imp_vals_sorted[_n//2]}, "
              f"p75={_imp_vals_sorted[3*_n//4]}, "
              f"max={_imp_vals_sorted[-1]}")
    print(f"Judgment importance: built in {time.time()-_ti:.1f}s")
else:
    print("[skip] idx_judgment_importance empty — graph not loaded.")

Built citation->doc_id map (2,158,211 entries)
Graph: 20,494,436 edges loaded, 3,155,263 skipped (cit not in corpus)
Graph: out-degree avg = 27.2, in-degree avg = 14.0
Graph: load time 96.2s
Judgment importance: 178,593 judgments, median=7, p75=44, max=204686
Judgment importance: built in 1.0s


# Phase 4 — Per-area bedrock + co-citation neighbours

## 4.1 Per-area bedrock — why we need it

For a query in legal area "criminal procedure", certain articles are
**universally cited** by every BGer detention decision (Art. 100 BGG,
Art. 42 BGG, Art. 66 BGG — the procedural/cost cluster). The LLM rarely
names these because they're "implicit" to lawyers but they're often gold.

Per-area bedrock is **corpus-derived**: from `legal_area_per_doc`, count
which canonical statutes appear most often in court rows of each area.

**Filter:** v6 added a critical fix — restrict per-area bedrock to canonicals
whose code matches one of the LLM-named codes (e.g., StPO + BGG for val_001).
v4 returned BGG-dominated lists across ALL areas because BGG appeal articles
are cited everywhere.

**Expected:** ~26 distinct legal areas; `criminal procedure and coercive measures`
top-8 should include 66 BGG, 78 BGG, 81 BGG.

**Failure modes:**
- `0 areas` → `legal_area_per_doc` is empty; check court enrichment has
  `legal_area_static` set.

In [ ]:
print("Building per-area bedrock index...")
_t = time.time()
per_area_canon_count = defaultdict(Counter)
for did, area in legal_area_per_doc.items():
    if not area or area == "law": continue
    canons = doc_statute_anchors.get(did, ())
    for canon in canons:
        per_area_canon_count[area][canon] += 1
print(f"  built in {time.time()-_t:.1f}s; areas: {len(per_area_canon_count)}")
for area in list(per_area_canon_count.keys())[:4]:
    print(f"  area={area!r}: top 8 = {per_area_canon_count[area].most_common(8)}")

# v7.5 NEW: derive code-family from corpus co-citation. For each pair of
# canons (a, b) co-cited in a court row, record the (code_a, code_b) pair
# weight. Used at retrieval time to expand statute_target_codes from
# LLM-named codes to corpus-related codes (no hardcoded statute cluster).
print("Building corpus-derived code-pair statistics...")
_tc = time.time()
code_pair_count = Counter()
for (a, b), n in co_citation_pairs.items():
    ca = a.split()[1] if " " in a else None
    cb = b.split()[1] if " " in b else None
    if ca and cb and ca != cb:
        code_pair_count[(ca, cb)] += n
        code_pair_count[(cb, ca)] += n   # symmetric
print(f"  code-pair statistics: {len(code_pair_count):,} directed pairs ({time.time()-_tc:.1f}s)")
print(f"  StPO's top related codes: {[(c, n) for (a, c), n in code_pair_count.most_common(2000) if a=='StPO'][:8]}")


Building per-area bedrock index...
  built in 2.0s; areas: 26
  area='constitutional and public law': top 8 = [('66 BGG', 13212), ('29 BV', 12189), ('89 BGG', 11479), ('82 BGG', 11464), ('42 BGG', 10836), ('9 BV', 9650), ('106 BGG', 9625), ('68 BGG', 8649)]
  area='administrative, tax, migration, and regulatory law': top 8 = [('42 BGG', 20301), ('106 BGG', 19460), ('66 BGG', 17471), ('105 BGG', 16033), ('95 BGG', 15722), ('83 BGG', 15231), ('68 BGG', 14674), ('89 BGG', 12187)]
  area='civil law': top 8 = [('63 OJ', 3189), ('8 ZGB', 2740), ('55 OJ', 2518), ('64 OJ', 2235), ('55 OG', 2085), ('63 OG', 1906), ('159 OG', 1876), ('9 BV', 1875)]
  area='criminal law and criminal procedure': top 8 = [('66 BGG', 28149), ('42 BGG', 20746), ('106 BGG', 19537), ('64 BGG', 13843), ('108 BGG', 12101), ('97 BGG', 12075), ('105 BGG', 11759), ('81 BGG', 11102)]


## 4.2 Co-citation neighbours — why we need it

For each statute target the LLM names (e.g., `Art. 221 StPO`), find the top-K
canonical statutes that are **cited together** in the same court rows most
often. Surfaces statute clusters that move together in legal practice.

**Filter:** drop neighbours that are TOO globally common (Art. 36 BV cited
in nearly every criminal case). 5000 = ~0.2% of 2.5 M corpus.

**Expected:** for `221 StPO` neighbours: `212 StPO`, `237 StPO`, `5 StPO`,
`5 EMRK` (the detention statute cluster). NOT `36 BV` (filtered out).

**Failure modes:**
- All-empty neighbours → `co_citation_pairs` was empty; check court enrichment
  contained statute_anchors.

In [ ]:
co_neighbours = defaultdict(list)
canon_count = Counter()
for did, canons in doc_statute_anchors.items():
    for c in canons: canon_count[c] += 1

for (a, b), cnt in co_citation_pairs.items():
    if cnt < CONFIG["co_citation_min_co_count"]: continue
    if canon_count[b] > CONFIG["co_citation_max_neighbour_count"]: pass  # may filter b
    if canon_count[a] > CONFIG["co_citation_max_neighbour_count"]: pass  # may filter a
    co_neighbours[a].append((b, cnt))
    co_neighbours[b].append((a, cnt))

# Apply frequency filter on neighbour side and keep top-K per source
co_neighbours = {
    src: sorted(
        ((nb, n) for nb, n in nbs if canon_count[nb] <= CONFIG["co_citation_max_neighbour_count"]),
        key=lambda x: -x[1]
    )[:CONFIG["co_citation_top_k_per_target"] * 2]
    for src, nbs in co_neighbours.items()
}
print(f"Co-citation neighbours indexed for {len(co_neighbours):,} canonicals "
      f"(after frequency filter: max global count = {CONFIG['co_citation_max_neighbour_count']}).")
print("Sample - neighbours of '221 StPO' AFTER frequency filter:")
for nb, n in co_neighbours.get("221 StPO", [])[:8]:
    print(f"  {nb}: co={n}, total_in_corpus={canon_count[nb]}")

Co-citation neighbours indexed for 2,282 canonicals (after frequency filter: max global count = 15000).
Sample - neighbours of '221 StPO' AFTER frequency filter:
  36 BV: co=915, total_in_corpus=7216
  31 BV: co=783, total_in_corpus=4339
  10 BV: co=731, total_in_corpus=5241
  212 StPO: co=714, total_in_corpus=1337
  221 BV: co=646, total_in_corpus=670
  237 StPO: co=487, total_in_corpus=1467
  5 BV: co=393, total_in_corpus=9685
  5 StPO: co=272, total_in_corpus=1955


# Phase 5 — BM25 (FTS5 in-memory)

## 5.1 What this cell does

Build SQLite FTS5 over `search_text[did]` for all 2.65 M docs. Provides
`bm25_search(query_text, k)` which returns top-k doc_ids ranked by BM25.

## 5.2 Why FTS5 (and not Whoosh / Lucene)

- Pure stdlib; no extra install.
- ~80 s build for 2.6 M short docs.
- Returns BM25-scored top-K in <50 ms.
- We can pass any expanded query text and get a stable ranking.

## 5.3 enhance() — corpus-derived query enrichment

Untitled75's reference notebook used a `enhance()` from train data that we
forbid. We replicate the IDEA (boost query with code names most associated
with query tokens) but train it on the **corpus** (laws_de) instead of train.

For each token in query, look up `tlf[token]` (a Counter mapping legal codes
to row counts). The top-K codes with highest score get appended to the query
multiple times, biasing BM25 toward law rows of those codes.

Example: query contains "detention" → boost `stpo` (high) more than `or`.

**Expected:** FTS5 build ~80 s. enhance() boost typically adds 5×5=25 token
repetitions to the query.

**Failure modes:**
- Slow build (>180 s) → swap MEMORY journal for OFF, or use a temp file.

In [ ]:
import sqlite3, math

# -----------------------------------------------------------------------------
# Phase 5.A — legacy single-language FTS (kept for back-compat / debugging)
# -----------------------------------------------------------------------------
print(f"Building in-memory FTS5 (legacy, single index) over {len(search_text):,} docs...")
_t = time.time()
_fts = sqlite3.connect(":memory:")
_fts.execute("PRAGMA journal_mode = MEMORY")
_fts.execute("PRAGMA synchronous = OFF")
_fts.execute("CREATE VIRTUAL TABLE docs USING fts5(did UNINDEXED, body, tokenize = 'unicode61 remove_diacritics 2')")
_inserted = 0
_batch = []
for did, txt in search_text.items():
    _batch.append((did, txt))
    if len(_batch) >= 50000:
        _fts.executemany("INSERT INTO docs(did, body) VALUES (?, ?)", _batch)
        _inserted += len(_batch); _batch.clear()
        if _inserted % 500000 == 0:
            print(f"  inserted {_inserted:,} ({time.time()-_t:.1f}s)")
if _batch:
    _fts.executemany("INSERT INTO docs(did, body) VALUES (?, ?)", _batch)
    _inserted += len(_batch)
_fts.commit()
print(f"FTS5 (legacy) built: {_inserted:,} rows in {time.time()-_t:.1f}s")


# -----------------------------------------------------------------------------
# Phase 5.B — per-language FTS5 indices
# -----------------------------------------------------------------------------
# Build one in-memory FTS5 per language. doc_language[did] was populated in
# cell 12 from the top-level `language` field (de/fr/it; anything else,
# including 'unknown' and missing, was normalised to 'en').
print("Building per-language FTS5 indices (de/fr/it/en)...")
_t_ml = time.time()
_LANGS = ("de", "fr", "it", "en")
_fts_by_lang = {}
_lang_doc_count = {}
for _L in _LANGS:
    _conn = sqlite3.connect(":memory:")
    _conn.execute("PRAGMA journal_mode = MEMORY")
    _conn.execute("PRAGMA synchronous = OFF")
    _conn.execute(
        "CREATE VIRTUAL TABLE docs USING fts5(did UNINDEXED, body, "
        "tokenize = 'unicode61 remove_diacritics 2')"
    )
    _fts_by_lang[_L] = _conn
    _lang_doc_count[_L] = 0

# Group inserts by language. Stream search_text once, route per doc_language.
_batches = {L: [] for L in _LANGS}
_unknown_lang_did = 0
for did, txt in search_text.items():
    L = doc_language.get(did, "en")
    if L not in _fts_by_lang:
        # safety net for any unexpected value (shouldn't happen after cell 12)
        L = "en"
        _unknown_lang_did += 1
    _batches[L].append((did, txt))
    _lang_doc_count[L] += 1
    if len(_batches[L]) >= 50000:
        _fts_by_lang[L].executemany(
            "INSERT INTO docs(did, body) VALUES (?, ?)", _batches[L]
        )
        _batches[L].clear()

for L in _LANGS:
    if _batches[L]:
        _fts_by_lang[L].executemany(
            "INSERT INTO docs(did, body) VALUES (?, ?)", _batches[L]
        )
        _batches[L].clear()
    _fts_by_lang[L].commit()

print(f"Per-language FTS5 built in {time.time()-_t_ml:.1f}s")
for _L in _LANGS:
    print(f"  fts[{_L}]: {_lang_doc_count[_L]:,} docs")
if _unknown_lang_did:
    print(f"  (note) {_unknown_lang_did:,} docs had no language tag and were "
          f"routed to 'en'")


# -----------------------------------------------------------------------------
# Phase 5.C — enhance() (corpus-derived BM25 lexicon expansion); unchanged.
# -----------------------------------------------------------------------------
def _enhance_codes(text):
    text_lc = text.lower()
    tokens = set()
    for tok in re.split(r"[^\w\d]+", text_lc, flags=re.UNICODE):
        if len(tok) >= CONFIG["bm25_min_token_len"]:
            tokens.add(tok)
    code_score = Counter()
    for tok in tokens:
        if tok not in tlf: continue
        n_docs = max(1, token_doc_count[tok])
        idf = math.log(1 + (max(1, len(search_text)) / n_docs))
        if idf < CONFIG["enhance_min_idf"]: continue
        for code, cnt in tlf[tok].most_common():
            code_score[code] += cnt * idf
    return [c for c, _ in code_score.most_common(CONFIG["enhance_top_k_codes"])]


# -----------------------------------------------------------------------------
# Phase 5.D — legacy bm25_search() (unchanged behaviour, single-index).
# -----------------------------------------------------------------------------
def bm25_search(query_text, k):
    boosted = _enhance_codes(query_text)
    enriched = query_text + " " + " ".join(c * CONFIG["enhance_repeat_count"] for c in boosted)
    fts_q = []
    for tok in re.split(r"[^\w\d]+", enriched, flags=re.UNICODE):
        if len(tok) >= CONFIG["bm25_min_token_len"]:
            fts_q.append(tok)
            if len(fts_q) >= CONFIG["bm25_max_query_terms"]: break
    if not fts_q: return []
    fts_query = " OR ".join(f'"{t}"' for t in fts_q)
    rows = _fts.execute(
        "SELECT did, bm25(docs) FROM docs WHERE docs MATCH ? ORDER BY bm25(docs) LIMIT ?",
        (fts_query, k),
    ).fetchall()
    return [(did, -score) for did, score in rows]


# -----------------------------------------------------------------------------
# Phase 5.E — NEW multi-language BM25 search.
# -----------------------------------------------------------------------------
# For each language L, builds a language-specific query string from:
#   - the original English query (always present; English query tokens still
#     match against English concepts_en mixed into court search_text rows)
#   - language-appropriate enrichment terms from the LLM-produced `targets`
# Runs FTS5 on that index, takes its top-k_per_lang. Merges by `did` taking
# max-score across languages, then returns the global top k_total.
#
# Score normalisation:
#   FTS5 bm25() returns a NEGATIVE score (more negative = more relevant). We
#   flip sign first (bigger = more relevant). Then divide by
#   log(corpus_size_for_lang + math.e) so a tiny language (~94k IT rows) and
#   a huge one (~1.4M DE rows) produce comparable score magnitudes.
def _build_lang_query(english_query, targets, lang):
    """Build the FTS5 query string for a given language."""
    parts = [english_query or ""]
    targets = targets or {}
    concept_en = list(targets.get("concept_targets_en") or [])
    term_de = list(targets.get("term_targets_de") or [])
    term_fr = list(targets.get("term_targets_fr") or [])
    if lang == "de":
        parts.extend(term_de)
        parts.extend(concept_en)
    elif lang == "fr":
        parts.extend(term_fr)
        parts.extend(concept_en)
    elif lang == "it":
        # No it-specific targets in the qexp schema; fall back to de+fr+en.
        parts.extend(term_de)
        parts.extend(term_fr)
        parts.extend(concept_en)
    else:  # "en" and any unexpected language
        parts.extend(concept_en)
    return " ".join(p for p in parts if p)


def _split_budget(k_total, lang_doc_count):
    """Allocate per-language budget proportional to docs in that language,
    with a small floor so tiny languages still contribute. Returns dict."""
    total_docs = sum(max(1, lang_doc_count[L]) for L in _LANGS)
    floor = max(1, k_total // 16)  # at least ~6% of budget per language
    raw = {L: max(floor, int(round(k_total * lang_doc_count[L] / total_docs)))
           for L in _LANGS}
    # Trim if floors caused over-allocation; never below the floor though.
    over = sum(raw.values()) - k_total
    if over > 0:
        # subtract from largest first
        for L in sorted(_LANGS, key=lambda x: -raw[x]):
            take = min(over, raw[L] - floor)
            if take <= 0: continue
            raw[L] -= take; over -= take
            if over <= 0: break
    return raw


def bm25_search_multilang(query, targets, k_total):
    """Run BM25 per language, merge by did with max-score, return top k_total.

    Args:
        query    : original English query string.
        targets  : dict from Phase 7 (statute_targets, term_targets_de,
                   term_targets_fr, concept_targets_en, ...).
        k_total  : total budget (CONFIG['budget_bm25']).

    Returns:
        list of (did, score) tuples, sorted by score desc, length <= k_total.
    """
    if not _fts_by_lang:
        return []
    budgets = _split_budget(k_total, _lang_doc_count)
    # enhance() runs on the original English query only (the corpus-derived
    # codes are language-agnostic statute codes like "stpo", "bgg" — these are
    # appended to every language's query).
    boosted = _enhance_codes(query or "")
    boost_str = " ".join(c * CONFIG["enhance_repeat_count"] for c in boosted)

    merged = {}  # did -> best normalised score
    for L in _LANGS:
        if _lang_doc_count[L] == 0:
            continue
        per_lang_query = _build_lang_query(query, targets, L)
        if boost_str:
            per_lang_query = per_lang_query + " " + boost_str
        # Tokenise for FTS5: alpha/num tokens >= min_len; cap to budget.
        fts_q = []
        seen = set()
        for tok in re.split(r"[^\w\d]+", per_lang_query, flags=re.UNICODE):
            if len(tok) < CONFIG["bm25_min_token_len"]:
                continue
            tl = tok.lower()
            if tl in seen:
                continue
            seen.add(tl)
            fts_q.append(tok)
            if len(fts_q) >= CONFIG["bm25_max_query_terms"]:
                break
        if not fts_q:
            continue
        fts_query = " OR ".join(f'"{t}"' for t in fts_q)
        # Cross-language comparability: divide raw score by log(N_lang + e).
        denom = math.log(_lang_doc_count[L] + math.e)
        try:
            rows = _fts_by_lang[L].execute(
                "SELECT did, bm25(docs) FROM docs WHERE docs MATCH ? "
                "ORDER BY bm25(docs) LIMIT ?",
                (fts_query, budgets[L]),
            ).fetchall()
        except sqlite3.OperationalError as _e:
            # malformed query (e.g. all stop-words) — skip this language.
            print(f"  bm25[{L}] skipped: {_e}")
            continue
        for did, raw_score in rows:
            # FTS5 bm25 is negative (lower = more relevant). Flip sign so
            # bigger = more relevant, then normalise by language size.
            norm = (-raw_score) / denom
            prev = merged.get(did)
            if prev is None or norm > prev:
                merged[did] = norm

    if not merged:
        return []
    out = sorted(merged.items(), key=lambda kv: -kv[1])[:k_total]
    return out


Building in-memory FTS5 (legacy, single index) over 2,649,348 docs...
  inserted 500,000 (11.4s)
  inserted 1,000,000 (27.1s)
  inserted 1,500,000 (44.0s)
  inserted 2,000,000 (61.3s)
  inserted 2,500,000 (77.5s)
FTS5 (legacy) built: 2,649,348 rows in 83.7s
Building per-language FTS5 indices (de/fr/it/en)...
Per-language FTS5 built in 70.0s
  fts[de]: 1,593,249 docs
  fts[fr]: 793,023 docs
  fts[it]: 125,583 docs
  fts[en]: 137,493 docs


# Phase 6 — Vector channel setup

## 6.1 What this cell does

Loads the 27 fp16 embedding chunks (~21 GB total) into a single `E_GPU`
tensor of shape (2.65 M, 4096). Provides `vector_search(q_emb, k)` doing
brute-force matmul on GPU.

## 6.2 Why brute-force GPU and not FAISS-IVF

- Blackwell has 95 GB VRAM; corpus E_GPU at fp16 fits in ~22 GB.
- Matmul `E_GPU @ q` is ~50 ms on GPU; comparable to FAISS-IVF query.
- No quantization recall loss, no index-build time.
- Per `personal_observations.md` Obs 3: dense embedding alone caps at
  R@1000 = 0.289 on val regardless of index — the bottleneck is "wrong kind
  of relationship for cosine similarity", not retrieval algorithm.

## 6.3 Manifest mapping

`qwen3_8b_unified_manifest.parquet` maps doc_id ↔ row_index in E_GPU.

**Expected:** 2.65 M rows in manifest, ~99.9% mapping coverage to our doc_ids.

**Failure modes:**
- `EMB_AVAILABLE = False` → vector channels skipped. Anchor + bedrock + BM25
  still run but recall drops.
- VRAM OOM during chunk concat → torch.cat allocates 2× peak; switch to
  in-place writes if chunk count grows.

In [ ]:
VECTOR_OK = False
E_GPU = None
my_did_for_row = None
row_for_did = None

if EMB_AVAILABLE:
    import torch as _torch, numpy as _np, pandas as _pd
    print("Loading manifest...")
    _t = time.time()
    _man = _pd.read_parquet(PATHS["emb_manifest"])
    print(f"  manifest rows: {len(_man):,}, cols: {list(_man.columns)}")
    # Build row_index -> our doc_id mapping
    row_for_did = {}; my_did_for_row = [None] * len(_man)
    for _, r in _man.iterrows():
        cit = r.get("citation") or ""
        fam = r.get("family") or ""
        ridx = int(r.get("row_index", -1))
        if ridx < 0: continue
        # Resolve to a doc_id in our cit_to_doc_ids based on family
        candidates = cit_to_doc_ids.get(cit, [])
        for d in candidates:
            if doc_meta.get(d, {}).get("family") == fam:
                row_for_did[d] = ridx
                if ridx < len(my_did_for_row):
                    my_did_for_row[ridx] = d
                break
    n_mapped = sum(1 for x in my_did_for_row if x is not None)
    print(f"  manifest->my_did mapping: {n_mapped:,}/{len(_man):,} ({100*n_mapped/len(_man):.1f}%) in {time.time()-_t:.1f}s")

    # Concat chunks to GPU
    print(f"  loading chunks to GPU (~21 GB)...")
    _t = time.time()
    _chunks = sorted(PATHS["emb_dir"].glob("qwen3_8b_unified_chunk*.npy"))
    arrs = []
    for cp in _chunks:
        arrs.append(_torch.from_numpy(_np.load(cp)).to("cuda", non_blocking=True))
    E_GPU = _torch.cat(arrs, dim=0); del arrs
    free_vram = (_torch.cuda.get_device_properties(0).total_memory
                 - _torch.cuda.memory_allocated()) / 1024**3
    print(f"  E_GPU shape={tuple(E_GPU.shape)} dtype={E_GPU.dtype}, "
          f"VRAM used={_torch.cuda.memory_allocated()/1024**3:.1f} GB, "
          f"free={free_vram:.1f} GB, "
          f"load {time.time()-_t:.1f}s")

    def vector_search(q_emb, k):
        if E_GPU is None: return []
        with _torch.no_grad():
            q = q_emb.to(E_GPU.device, dtype=E_GPU.dtype)
            q = q / (q.norm(dim=-1, keepdim=True) + 1e-9)
            scores = E_GPU @ q
            top_v, top_i = _torch.topk(scores, k=min(k, scores.shape[0]))
        out = []
        for s, i in zip(top_v.cpu().tolist(), top_i.cpu().tolist()):
            d = my_did_for_row[i]
            if d is not None: out.append((d, float(s)))
        return out

    VECTOR_OK = True
else:
    print("[skip] EMB_AVAILABLE = False; vector channels will be skipped.")
    def vector_search(q_emb, k): return []

Loading manifest...
  manifest rows: 2,652,248, cols: ['doc_id', 'family', 'citation', 'row_index']
  manifest->my_did mapping: 2,649,348/2,652,248 (99.9%) in 40.1s
  loading chunks to GPU (~21 GB)...
  E_GPU shape=(2652248, 4096) dtype=torch.float16, VRAM used=20.2 GB, free=74.7 GB, load 460.8s


## 6.4 Embedding model for query

**What:** Load Qwen3-Embedding-8B via SentenceTransformer for **query-side**
encoding (corpus-side is pre-encoded).

**Why:** We need the same model that produced the corpus embeddings, with
its canonical instruction prefix.

**VRAM:** ~16 GB. Loaded after Qwen3-32B is freed (Phase 7).

**Failure modes:**
- HF download blocked → preflight `huggingface_hub.snapshot_download` once.
- Model weights mismatch (8B vs 8B-Embedding) → ensure model id is exactly
  `Qwen/Qwen3-Embedding-8B`.

In [ ]:
EMB_MODEL = None
def encode_query(text):
    return EMB_MODEL.encode(
        [text],
        prompt_name="query",
        convert_to_tensor=True,
        normalize_embeddings=True,
    )[0]

# Phase 7 — Query expansion (Qwen3-32B → JSON targets)

## 7.1 What this cell does

Load Qwen3-32B in bf16 (~65 GB VRAM), prompt it with a fixed schema, parse
the JSON output. Then **free the model from VRAM** so Qwen3-Embedding-8B and
the corpus E_GPU can fit.

## 7.2 Why structured JSON expansion

Plain HyDE (generate fake answer, embed it) was tried in v5 and added zero
measurable recall. Structured expansion gives us:

- `statute_targets`: list of "Art. N CODE" strings → fed to `law_direct_match`,
  `court_statute`, `co_citation`, `per_area_bedrock` filter.
- `concept_targets_en`: list of English legal concepts → fed to `concept_en`.
- `term_targets_de` / `term_targets_fr`: original-language terms → fed to
  `term_orig` and BM25.
- `legal_area_keywords`: 1-5 legal-area phrases → fed to `per_area_bedrock`.
- `case_targets`: BGE/docket case targets (NOT used as anchor channel because
  v5 measured: LLM hallucinates BGE numbers in every prior run; channel dropped).

## 7.3 Prompt design — what's in / out

- **Few-shot examples** of detention queries to keep 32B on standard Swiss
  legal vocabulary (e.g. `Untersuchungshaft`, not "preventive incarceration").
- **No invented citations** instruction — we trust statute_targets only as
  guidance for canonicalization, never as proof.
- **No legal advice** style — the model just enumerates entities.

## 7.4 Expected outputs

For val_001 (pre-trial detention extension):
- statute_targets: 4-8 items including `Art. 221 StPO`, `Art. 100 BGG`
- concept_targets_en: 15-25 items including `preventive detention`, `collusion risk`,
  `proportionality`
- term_targets_de: 15-25 items including `Untersuchungshaft`, `Kollusionsgefahr`,
  `Verhältnismässigkeit`
- legal_area_keywords: ~5 items

## 7.5 Failure modes

- Model emits `<think>` chain-of-thought and runs out of tokens → set
  `enable_thinking=False`, raise `qwen_max_new_tokens`.
- JSON parse fails → fallback regex catches partial. If still empty, drop
  to a hand-crafted target dict (don't crash).

In [ ]:
QEXP_PROMPT_SYSTEM = (
    "You are a Swiss legal-research assistant. You do NOT invent citations. "
    "You output ONLY JSON matching the schema."
)

QEXP_PROMPT_USER = '''Given an English legal query about Swiss law, produce a JSON
object with these EXACT keys:

{
  "statute_targets":      [list of "Art. N CODE" strings, codes from {StPO, StGB, BGG, ZGB, OR, ZPO, BV, EMRK, IPRG, IRSG, ...}],
  "case_targets":         [list of "BGE V D P" or "1B_N/Y" docket strings],
  "concept_targets_en":   [list of English legal concept phrases],
  "term_targets_de":      [list of original-language German terms],
  "term_targets_fr":      [list of French legal terms],
  "legal_area_keywords":  [1-5 short legal-area phrases like "criminal procedure", "detention law"]
}

Examples for "May a court extend pre-trial detention under StPO":
- statute_targets: ["Art. 221 StPO", "Art. 222 StPO", "Art. 227 StPO", "Art. 100 BGG"]
- concept_targets_en: ["preventive detention", "collusion risk", "proportionality"]
- term_targets_de: ["Untersuchungshaft", "Kollusionsgefahr", "Verhältnismässigkeit"]
- term_targets_fr: ["détention provisoire", "danger de collusion", "proportionnalité"]
- legal_area_keywords: ["criminal procedure", "detention law", "proportionality principle"]

Output ONLY the JSON. Query:
{QUERY}
'''

import time
import re as _re
import json as _json
import torch as _torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print(f"[qexp] loading {CONFIG['qwen_query_model']} (~65 GB bf16)...")
_t = time.time()

qtok = AutoTokenizer.from_pretrained(CONFIG["qwen_query_model"])

qmod = AutoModelForCausalLM.from_pretrained(
    CONFIG["qwen_query_model"],
    dtype=_torch.bfloat16,          # use torch_dtype=_torch.bfloat16 if your transformers version requires it
    device_map="auto",
)

qmod.eval()

print(f"[qexp] model loaded in {time.time() - _t:.1f}s")

# Build chat-formatted prompt
_messages = [
    {"role": "system", "content": QEXP_PROMPT_SYSTEM},
    {"role": "user", "content": QEXP_PROMPT_USER.replace("{QUERY}", QUERY)},
]

_inp = qtok.apply_chat_template(
    _messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
    enable_thinking=False,
)

# With device_map="auto", send inputs to the model's first real device.
_input_device = next(p.device for p in qmod.parameters() if p.device.type != "meta")
_inp = {k: v.to(_input_device) for k, v in _inp.items()}

with _torch.no_grad():
    _out = qmod.generate(
        **_inp,
        max_new_tokens=CONFIG["qwen_max_new_tokens"],
        do_sample=False,
        pad_token_id=qtok.eos_token_id,
    )

_prompt_len = _inp["input_ids"].shape[1]
_resp = qtok.decode(_out[0][_prompt_len:], skip_special_tokens=True)

print(f"RAW (first 400 chars): {_resp[:400]}")

# Free Qwen3-32B from VRAM
del qmod, qtok, _inp, _out

_torch.cuda.empty_cache()
if _torch.cuda.is_available():
    _torch.cuda.synchronize()
    print(f"[qexp] freed Qwen3-32B; CUDA mem={_torch.cuda.memory_allocated() / 1024**3:.1f} GB")
else:
    print("[qexp] freed Qwen3-32B")

# Parse JSON
_match = _re.search(r"\{.*\}", _resp, flags=_re.S)

targets = {}
if _match:
    try:
        targets = _json.loads(_match.group(0))
    except Exception as e:
        print(f"[qexp] JSON parse failed: {e}; using empty targets")

for k in (
    "statute_targets",
    "case_targets",
    "concept_targets_en",
    "term_targets_de",
    "term_targets_fr",
    "legal_area_keywords",
):
    targets.setdefault(k, [])

print()
print("Targets parsed:")
for k, v in targets.items():
    print(f"  {k}: ({len(v)}) {v[:5]}")

[qexp] loading Qwen/Qwen3-32B (~65 GB bf16)...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[qexp] model loaded in 201.6s
RAW (first 400 chars): {
  "statute_targets": ["Art. 221 StPO", "Art. 222 StPO", "Art. 227 StPO", "Art. 100 BGG", "Art. 101 BGG"],
  "case_targets": ["BGE 147 II 123", "BGE 145 II 345", "1B_123/2024"],
  "concept_targets_en": ["preventive detention", "collusion risk", "proportionality", "investigative necessity", "witness tampering"],
  "term_targets_de": ["Untersuchungshaft", "Kollusionsgefahr", "Verhältnismässigkeit",
[qexp] freed Qwen3-32B; CUDA mem=20.2 GB

Targets parsed:
  statute_targets: (5) ['Art. 221 StPO', 'Art. 222 StPO', 'Art. 227 StPO', 'Art. 100 BGG', 'Art. 101 BGG']
  case_targets: (3) ['BGE 147 II 123', 'BGE 145 II 345', '1B_123/2024']
  concept_targets_en: (5) ['preventive detention', 'collusion risk', 'proportionality', 'investigative necessity', 'witness tampering']
  term_targets_de: (5) ['Untersuchungshaft', 'Kollusionsgefahr', 'Verhältnismässigkeit', 'Ermittlungsbedürftigkeit', 'Zeugenbeeinflussung']
  term_targets_fr: (5) ['détentio

## 7.6 Encode query (raw + enriched)

**What:** Load Qwen3-Embedding-8B (now that 32B is freed), encode the query
twice — once raw, once with concept/term keywords appended.

**Why two embeddings:** Different recall surfaces.
- Raw query embedding captures the LITERAL sentence semantics.
- Enriched query (raw + 60 keywords) shifts the embedding toward the legal
  domain and matches more domain-aligned corpus rows.

**Expected:** Two 4096-dim normalized vectors. Enriched length ~2300 chars.

**Failure modes:**
- VRAM OOM → ensure Qwen3-32B is fully freed; check `torch.cuda.empty_cache()`
  was called.

In [ ]:
print("Loading Qwen3-Embedding-8B...")
from sentence_transformers import SentenceTransformer
EMB_MODEL = SentenceTransformer(CONFIG["vector_emb_model"])
print(f"  done")

print("Encoding raw query...")
q_emb_raw = encode_query(QUERY)
print("Encoding enriched query...")
enriched_bits = (
    (targets.get("term_targets_de") or [])
  + (targets.get("term_targets_fr") or [])
  + (targets.get("concept_targets_en") or [])
)
enriched_query = QUERY + " " + " ".join(enriched_bits[:60])
q_emb_enriched = encode_query(enriched_query)
print(f"  enriched query length: {len(enriched_query)} chars, "
      f"keywords appended: {min(60, len(enriched_bits))}")

Loading Qwen3-Embedding-8B...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

  done
Encoding raw query...
Encoding enriched query...
  enriched query length: 1367 chars, keywords appended: 15


# Phase 8 — Channels

## 8.1 What this section does

Defines all retrieval channels as standalone functions, runs them, prints
per-channel R@K (recall against val_001 gold).

## 8.2 The 14 channels and which gap each closes

| # | Channel | Index used | Gap closed |
|---|---|---|---|
| 1 | `law_direct_match` | `idx_law_direct` | Statute targets named by LLM → matching law rows. Uncapped. |
| 2 | `court_statute` | `idx_court_statute` | LLM-named statute appears in row's `statute_anchors`. |
| 3 | `co_citation` | `co_neighbours` + `idx_law_direct` + `idx_court_statute` | Statute cluster co-cited with LLM target (e.g., 221 StPO + 212 StPO). |
| 4 | `per_area_bedrock` | `per_area_canon_count` filtered by LLM codes | Universal procedural articles per legal area. |
| 5 | `statute_backprop` | `doc_statute_anchors` | Caught court rows → law articles they cite (procedural cluster). |
| 6 | `sibling_expansion` | `idx_court_base` | All Es of caught judgments (own-judgment fan-out). v6 budget bug fixed. |
| 7 | **`graph_forward`** (v7 NEW) | `idx_graph_out` | Caught row → text-cited targets + case-level fan-out across judgments. |
| 8 | **`graph_reverse`** (v7 NEW) | `idx_graph_in` | Caught row ← rows that text-cite it (co-citing peers). |
| 9 | **`graph_2hop`** (v7 NEW, off by default) | `idx_graph_out` | 2-hop forward expansion. Disable unless Phase 10 says to enable. |
| 10 | `concept_en` | `idx_concept_en` | LLM concepts → English-tagged rows. |
| 11 | `term_orig` | `idx_term_orig` | LLM DE/FR terms → original-language-tagged rows. |
| 12 | `bm25` | FTS5 + enhance() | Lexical match including code-name boosts. |
| 13 | `vector_raw` | `E_GPU` brute-force | Dense semantic match on raw query. |
| 14 | `vector_enriched` | `E_GPU` brute-force | Dense semantic match on keyword-enriched query. |

## 8.3 Channel function definitions

In [ ]:
from collections import Counter

def channel_law_direct(canon_set, idx, budget):
    counter = Counter()
    for canon in canon_set:
        for did in idx[canon]: counter[did] += 1
    items = counter.most_common()
    return items if budget is None else items[:budget]

def channel_court_statute(statute_canons, idx, idx_count=None, doc_meta=None, budget=None):
    """Score court rows for an LLM-named statute canonical set.

    Score(did) = (sum over matched canons of 1/log(2 + global_count[canon]))
                 * (1 + 0.3 * (n_matches - 1))         # multi-match bonus
                 * paragraph_role_weight(did)          # 0.4 / 0.6 / 1.0 / 1.5

    Backwards-compatible: idx_count and doc_meta are optional. If idx_count is
    None (e.g. cell 12 hasn't been re-run with the new patch), counts are derived
    from idx on the fly. If doc_meta is None, role weighting is skipped (1.0).
    """
    import math as _math
    if idx_count is None:
        idx_count = {c: len(idx[c]) for c in statute_canons if c in idx}

    # paragraph_role -> multiplicative weight
    _ROLE_W = {
        "legal_standard": 1.5, "reasoning": 1.5,
        "application":    1.5, "holding":   1.5,
        "facts":              1.0, "procedural_history": 1.0,
        "citation":           1.0, "neutral_default":    1.0,
        "costs":        0.6, "disposition": 0.6, "notification": 0.6,
        "neutral":      0.4,
    }

    # Per-doc accumulator: {did: [matches_so_far, summed_specificity_weight]}
    per_doc = {}
    for canon in statute_canons:
        dids = idx.get(canon)
        if not dids:
            continue
        cnt = idx_count.get(canon)
        if cnt is None:
            cnt = len(dids)
        w_canon = 1.0 / _math.log(2 + cnt)
        for did in dids:
            slot = per_doc.get(did)
            if slot is None:
                per_doc[did] = [1, w_canon]
            else:
                slot[0] += 1
                slot[1] += w_canon

    if not per_doc:
        return []

    scored = []
    for did, (n_matches, base_w) in per_doc.items():
        score = base_w * (1.0 + 0.3 * (n_matches - 1))
        if doc_meta is not None:
            meta = doc_meta.get(did) or {}
            role = meta.get("paragraph_role")
            if role is None or role == "":
                rw = 0.4
            else:
                rw = _ROLE_W.get(role, 1.0)
            score *= rw
        scored.append((did, float(score)))

    # Stable deterministic ordering: descending score, ascending doc_id on ties.
    scored.sort(key=lambda x: (-x[1], x[0]))
    if budget is None:
        return scored
    return scored[:budget]

import math as _math_v74

_SUBSTANTIVE_ROLES = {"reasoning", "legal_standard", "application", "holding"}

def _judgment_factor(cb, idx_judgment_importance):
    """sqrt(1 + log(1 + importance)). Default 1.0 for unknown / 0 importance."""
    imp = idx_judgment_importance.get(cb, 0) if cb else 0
    if imp <= 0:
        return 1.0
    return _math_v74.sqrt(1.0 + _math_v74.log(1.0 + float(imp)))

def _role_boost(doc_meta, did):
    m = doc_meta.get(did) or {}
    role = (m.get("paragraph_role") or "").strip().lower()
    return 1.5 if role in _SUBSTANTIVE_ROLES else 1.0

def channel_sibling(seed_doc_ids, idx_court_base, idx_judgment_importance,
                    doc_meta, budget):
    # 1) seed_count[cb] = how many caught seeds share court_base cb
    seed_count = Counter()
    for did in seed_doc_ids:
        m = doc_meta.get(did) or {}
        cb = m.get("court_base")
        if cb: seed_count[cb] += 1
    # 2) emit every sibling row with score = seed_count * judgment_factor * role_boost
    seed_set = set(seed_doc_ids)
    scored = {}
    for cb, cnt in seed_count.items():
        factor = _judgment_factor(cb, idx_judgment_importance)
        for s in idx_court_base.get(cb, ()):
            if s in seed_set: continue
            sc = float(cnt) * factor * _role_boost(doc_meta, s)
            # keep best score per doc (a row only belongs to one cb)
            if sc > scored.get(s, 0.0):
                scored[s] = sc
    # 3) deterministic order: score desc, doc_id asc
    items = sorted(scored.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]

def channel_graph_forward(seed_doc_ids, idx_graph_out, idx_judgment_importance,
                          doc_meta, budget):
    # edge-count per target (how many distinct seeds point to it)
    seed_set = set(seed_doc_ids)
    edge_count = Counter()
    for did in seed_doc_ids:
        for t in idx_graph_out.get(did, ()):
            edge_count[t] += 1
    for d in seed_set:
        edge_count.pop(d, None)
    # weight each target by importance of its OWN judgment + role boost
    scored = {}
    for t, cnt in edge_count.items():
        m = doc_meta.get(t) or {}
        cb_of_t = m.get("court_base")
        factor = _judgment_factor(cb_of_t, idx_judgment_importance)
        sc = float(cnt) * factor * _role_boost(doc_meta, t)
        scored[t] = sc
    items = sorted(scored.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]

def channel_graph_reverse(seed_doc_ids, idx_graph_in, idx_judgment_importance,
                          doc_meta, budget,
                          landmark_imp_threshold=5):
    """Reverse-graph expansion gated to landmark seeds.
    Only follows incoming-edges from seeds whose judgment has
    importance >= landmark_imp_threshold (default 5). Each contributing
    edge is weighted by the SEED-judgment importance, so landmark seeds
    dominate.
    """
    seed_set = set(seed_doc_ids)
    scored = {}
    for did in seed_doc_ids:
        m = doc_meta.get(did) or {}
        cb_seed = m.get("court_base")
        imp_seed = idx_judgment_importance.get(cb_seed, 0) if cb_seed else 0
        if imp_seed < landmark_imp_threshold:
            continue
        factor = _judgment_factor(cb_seed, idx_judgment_importance)
        for s in idx_graph_in.get(did, ()):
            if s in seed_set: continue
            inc = factor * _role_boost(doc_meta, s)
            scored[s] = scored.get(s, 0.0) + inc
    items = sorted(scored.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]

def channel_graph_2hop(seed_doc_ids, idx_graph_out, budget):
    seed_set = set(seed_doc_ids)
    intermediate = set()
    for did in seed_doc_ids:
        intermediate.update(idx_graph_out.get(did, ()))
    intermediate -= seed_set
    counter = Counter()
    for x in intermediate:
        for t in idx_graph_out.get(x, ()):
            if t in seed_set: continue
            counter[t] += 1
    for d in intermediate:
        counter.pop(d, None)
    return counter.most_common(budget)

_CONCEPT_STOPWORDS_EN = frozenset({
    "a","an","the","of","in","on","at","by","for","with","to","from",
    "and","or","but","is","are","was","were","be","been","being",
    "has","have","had","do","does","did","no","not",
})
_CONCEPT_TOKEN_SPLIT_RE = re.compile(r"[^\w]+", re.UNICODE)

def _concept_tokens(s):
    """Whitespace+punct split, lowercase. Returns full token list."""
    if not s: return []
    return [t for t in _CONCEPT_TOKEN_SPLIT_RE.split(s.lower()) if t]

def _concept_meaningful_tokens(s):
    return [t for t in _concept_tokens(s) if t not in _CONCEPT_STOPWORDS_EN]

def expand_concepts_weighted(llm_concepts, corpus_concept_keys, top_k=25):
    """Replace strict-substring matcher with a weighted scorer.

    Combines three signals against each corpus concept:
      1. Exact match               -> weight 1.0
      2. Substring (either dir)    -> weight 0.85 * shorter/longer
         (must also share >=1 meaningful token; otherwise a stopword
         only embedding like 'of detention' could hijack 'of').
      3. Token overlap             -> weight shared/max(q_tok, c_tok)
         after stripping English stopwords from BOTH sides.

    Returns list[(corpus_concept, weight)] sorted by weight desc,
    capped at top_k per query concept (max weight kept across
    multiple query concepts hitting the same corpus concept).
    """
    W_EXACT = 1.0
    W_SUBSTR_MAX = 0.85  # capped below exact so true matches dominate

    keys = list(corpus_concept_keys)
    # Pre-tokenize the corpus once: corpus_concept -> (set_of_tokens, n_tokens)
    corpus_meaningful = {}
    for k in keys:
        m = _concept_meaningful_tokens(k)
        if m:
            corpus_meaningful[k] = (set(m), len(m))

    best_weight = {}  # corpus_concept -> max weight across query concepts

    for raw in llm_concepts or []:
        c = (raw or "").lower().strip()
        if not c or len(c) < 4:
            continue
        c_meaningful = _concept_meaningful_tokens(c)
        if not c_meaningful:
            # Pure-stopword query (e.g. "of the") yields no matches.
            continue
        c_set = set(c_meaningful)
        c_len = len(c_meaningful)
        c_chars = len(c)

        per_query = []  # (weight, length_diff, corpus_concept)

        # 1. Exact match
        if c in corpus_concept_keys:
            per_query.append((W_EXACT, 0, c))

        # 2. Substring match (either direction), gated on shared
        #    meaningful token to block stopword-only bridges.
        for cv in keys:
            if cv == c:
                continue
            if c in cv or cv in c:
                cv_info = corpus_meaningful.get(cv)
                if not cv_info:
                    continue
                cv_set, cv_len = cv_info
                if not (c_set & cv_set):
                    # only stopword/character overlap; reject
                    continue
                shorter = min(c_chars, len(cv))
                longer  = max(c_chars, len(cv))
                if longer <= 0:
                    continue
                w = W_SUBSTR_MAX * (shorter / longer)
                per_query.append((w, abs(len(cv) - c_chars), cv))

        # 3. Token overlap (stopwords already stripped on both sides)
        for cv, (cv_set, cv_len) in corpus_meaningful.items():
            if cv == c:
                continue
            shared = c_set & cv_set
            if not shared:
                continue
            denom = max(c_len, cv_len)
            if denom <= 0:
                continue
            w = len(shared) / denom
            per_query.append((w, abs(cv_len - c_len), cv))

        # Dedupe within this query concept: max weight per corpus key.
        local_best = {}
        for w, ld, cv in per_query:
            cur = local_best.get(cv)
            if cur is None or w > cur[0] or (w == cur[0] and ld < cur[1]):
                local_best[cv] = (w, ld)

        # Rank: weight desc, then length-diff asc, then alphabetical.
        ranked = sorted(local_best.items(),
                        key=lambda kv: (-kv[1][0], kv[1][1], kv[0]))
        for cv, (w, _) in ranked[:top_k]:
            prior = best_weight.get(cv, 0.0)
            if w > prior:
                best_weight[cv] = w

    # Final ordering: weight desc, then alphabetical for determinism.
    return sorted(best_weight.items(), key=lambda kv: (-kv[1], kv[0]))

def channel_concept(expanded_weighted, idx, budget):
    """Consume list[(concept, weight)] from expand_concepts_weighted.

    Each doc accumulates the sum of weights from every matched
    corpus concept it carries. Top `budget` by accumulated score.
    Backward-compatible: tolerates a list of plain strings (legacy)
    by treating each as weight=1.0.
    """
    scores = {}
    for item in expanded_weighted or []:
        if isinstance(item, tuple):
            tok, w = item
        else:
            tok, w = item, 1.0
        if not tok:
            continue
        for did in idx.get(tok, ()):
            scores[did] = scores.get(did, 0.0) + float(w)
    if not scores:
        return []
    # Sort by score desc, then doc_id for determinism. Cap at budget.
    items = sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))
    if budget is not None:
        items = items[:budget]
    return items

def channel_term(targets, idx, idx_lemma, budget,
                 corpus_keys=None, lemma_score=0.7,
                 min_substring_len=4):
    """Term_orig channel — exact + lemma + substring fused scorer.

    Score model (per query-term/corpus-term pair, MAX over paths):
      exact         -> 1.0
      lemma equal   -> `lemma_score` (default 0.7) (DE-only; we still try FR/IT
                       through term_lemma but lemma_score is conservative)
      Q in T        -> len(Q) / len(T)
      T in Q        -> len(T) / len(Q)   (only if len(T) >= min_substring_len)
    Per doc_id we sum scores across all matching pairs and return the
    top-`budget` doc_ids by sum.
    """
    if corpus_keys is None:
        # Fallback: derive surface-form keys from the exact-match index. Slower
        # to construct on the fly but keeps callers without the precomputed set
        # working.
        corpus_keys = list(idx.keys())
    else:
        corpus_keys = list(corpus_keys)

    # Collect normalized query terms (DE first, then FR; we treat both
    # symmetrically — substring matching is language-agnostic, lemma logic
    # is most reliable for DE but doesn't actively hurt FR/IT because the
    # lemma function is identity for tokens with no removable suffix).
    q_terms = []
    for key in ("term_targets_de", "term_targets_fr"):
        for raw in targets.get(key, []) or []:
            tok = norm_token(raw, CONFIG["lowercase_terms"])
            if tok:
                q_terms.append(tok)
    if not q_terms:
        return []

    # doc_id -> accumulated score (max-per-pair, summed across query terms)
    score = Counter()

    for Q in q_terms:
        Q_lemma = term_lemma(Q)
        Q_len = len(Q)
        # Per-doc max for this single query term — prevents two paths
        # (e.g. exact + lemma) from double-counting the same doc.
        per_q = {}
        def _bump(did, s):
            if s > per_q.get(did, 0.0):
                per_q[did] = s

        # 1. Exact match — strongest signal, score 1.0.
        if Q in idx:
            for did in idx[Q]:
                _bump(did, 1.0)

        # 2. Lemma match — only fire when lemma differs from surface or when
        #    Q's lemma keys are present. Score 0.7 (capped below exact).
        if Q_lemma and Q_lemma in idx_lemma:
            for did in idx_lemma[Q_lemma]:
                _bump(did, lemma_score)

        # 3. Substring scan — for each corpus term T, compute the longer
        #    of Q⊂T and T⊂Q. Skip the exact-equal case (already scored).
        if Q_len >= 1:
            for T in corpus_keys:
                if T == Q:
                    continue
                T_len = len(T)
                s = 0.0
                if Q in T:
                    s = Q_len / T_len      # len(Q) / len(T)
                elif T_len >= min_substring_len and T in Q:
                    s = T_len / Q_len      # len(T) / len(Q)
                if s <= 0.0:
                    continue
                # Each corpus term T may map to many doc_ids; bump them all.
                for did in idx.get(T, ()):  # exact-form posting is canonical
                    _bump(did, s)

        # Roll the per-Q max scores into the cross-Q sum.
        for did, s in per_q.items():
            score[did] += s

    # most_common-style ordering by score, with deterministic tie-break on doc_id.
    items = sorted(score.items(), key=lambda kv: (-kv[1], kv[0]))
    return items if budget is None else items[:budget]

def channel_per_area_bedrock(legal_area_keywords, statute_target_codes,
                             per_area_canon_count, idx_law_direct, budget):
    if not legal_area_keywords: return []
    keys = [k.lower() for k in legal_area_keywords]
    selected_areas = set()
    for area in per_area_canon_count.keys():
        for k in keys:
            if k in area: selected_areas.add(area); break
    if not selected_areas: return []
    canon_score = Counter()
    for area in selected_areas:
        for canon, n in per_area_canon_count[area].most_common(CONFIG["per_area_top_n"]):
            canon_score[canon] = max(canon_score[canon], n)
    out = []; seen = set()
    for canon, _ in canon_score.most_common():
        canon_code = canon.split()[1] if canon and " " in canon else None
        if statute_target_codes and canon_code not in statute_target_codes:
            continue
        for did in idx_law_direct.get(canon, set()):
            if did not in seen:
                out.append((did, canon_score[canon])); seen.add(did)
        if len(out) >= budget: break
    return out[:budget]

def channel_statute_backprop(seed_court_dids, doc_statute_anchors, idx_law_direct, budget):
    import math as _math
    canon_to_courts = defaultdict(set)
    for did in seed_court_dids:
        for canon in doc_statute_anchors.get(did, set()):
            canon_to_courts[canon].add(did)
    # Specificity weighting: rare canons (small global court count) score more per
    # caught row than common procedural canons (Art. 100 BGG, Art. 9 BV, ...).
    # Idempotent against court_statute fix agent which may also define this:
    _idx_canon_ct = (locals().get('idx_court_statute_count')
                     or globals().get('idx_court_statute_count')
                     or {c: len(s) for c, s in idx_court_statute.items()})
    counter = Counter()
    for canon, court_set in canon_to_courts.items():
        n_caught = len(court_set)
        global_ct = _idx_canon_ct.get(canon, n_caught)
        score = n_caught * (1.0 / _math.log(2 + global_ct))
        for law_did in idx_law_direct.get(canon, set()):
            if counter[law_did] < score: counter[law_did] = score
    return counter.most_common(budget)

def channel_co_citation(targets, co_neighbours, idx_law_direct, idx_court_statute, budget):
    import math as _math
    # Use canon_count from cell 20 if available; otherwise derive from idx_court_statute.
    _canon_ct = (locals().get('canon_count')
                 or globals().get('canon_count')
                 or {c: len(s) for c, s in idx_court_statute.items()})
    counter = Counter()
    for raw in targets.get("statute_targets", []) or []:
        canon = statute_anchor_canonical(raw)
        if not canon: continue
        for nb, n in co_neighbours.get(canon, []):
            spec = 1.0 / _math.log(2 + _canon_ct.get(nb, 1))
            score = n * spec
            for did in idx_law_direct.get(nb, set()):
                if counter[did] < score: counter[did] = score
            for did in idx_court_statute.get(nb, set()):
                if counter[did] < score: counter[did] = score
    return counter.most_common(budget)

def channel_vector(q_emb, k):
    return vector_search(q_emb, k)

## 8.4 Run all channels + per-channel diagnostics

**What:** Calls every channel function, builds the seed pool from topical
hits, runs sibling + graph + statute_backprop with the seed.

**Diagnostic table:** for every channel, prints:
- size: actual hits returned (≤ budget)
- gold_in_ch: how many of val_001's 42 gold are in this channel's hits
- recall: gold_in_ch / 42

Plus a UNION row showing the upper bound on R@K — no fusion can exceed this.

**Expected for val_001:**
- vector_raw / vector_enriched: ~9-12 gold (best topical channels per Obs 3)
- concept_en: ~9 gold
- statute_backprop: ~8 gold
- law_direct_match: ~7 gold
- bm25: ~3-7 gold
- sibling_expansion: depends on seed; with v7 budget=2000, expect 4-8 sibling-Es
- graph_forward: should add 4-8 NEW gold not in topical channels (case-level fan-out)
- graph_reverse: 1-3 gold (co-citing peer rows that contain gold)
- UNION upper bound: aim for ≥ 28/42 = 67% (was 23/42 = 55% in v6)

**Failure modes:**
- UNION drops vs v6 → graph channels misconfigured (check Phase 3 mapping).
- A channel that should produce 100s of hits returns 0 → its index is empty;
  go back and verify the build step.

In [ ]:
# Build canonical sets from LLM targets
llm_statute_canons = set()
for raw in targets.get("statute_targets", []) or []:
    c = statute_anchor_canonical(raw)
    if c: llm_statute_canons.add(c)
co_expanded_canons = set(llm_statute_canons)
for canon in llm_statute_canons:
    for nb, _ in co_neighbours.get(canon, []):
        co_expanded_canons.add(nb)
print(f"Statute canons: LLM={len(llm_statute_canons)}, +co-citation={len(co_expanded_canons)}")

# v7.5: expand statute target codes with corpus-derived related codes.
# Starting from LLM-named codes (e.g., {StPO, BGG}), find codes that
# co-occur most frequently in the same court rows (e.g., StPO -> {StBOG,
# BV, EMRK, StGB, ZGB, ...}). Pure corpus statistic, not hardcoded.
statute_target_codes = set()
for canon in llm_statute_canons:
    if " " in canon: statute_target_codes.add(canon.split()[1])

llm_codes_only = set(statute_target_codes)
_kfam = CONFIG.get("code_family_top_k", 8)
for c in llm_codes_only:
    related = sorted(
        ((cc, n) for (a, cc), n in code_pair_count.items() if a == c),
        key=lambda x: -x[1])[:_kfam]
    for cc, _ in related:
        statute_target_codes.add(cc)
print(f"LLM target codes: {sorted(llm_codes_only)}")
print(f"Code-family expanded ({_kfam}/code from corpus co-citation): {sorted(statute_target_codes - llm_codes_only)}")

# Concept expansion (strict-substring against corpus vocab).
llm_concepts = (targets.get("concept_targets_en") or []) + (targets.get("legal_area_keywords") or [])
corpus_concept_keys = set(idx_concept_en.keys())
expanded_weighted = expand_concepts_weighted(llm_concepts, corpus_concept_keys, top_k=25)
print(f"Concepts: LLM={len(llm_concepts)} -> expanded={len(expanded_weighted)}")

# Diagnostic alias for Phase 11. expanded_weighted is list[(concept, weight)];
# Phase 11's concepts_pointing_to() iterates strings, so flatten.
expanded_concepts = [c for c, _ in expanded_weighted] if expanded_weighted else []

# Topical channels
ch_law_direct = channel_law_direct(co_expanded_canons, idx_law_direct, CONFIG["budget_law_direct"])
ch_court_stat = channel_court_statute(llm_statute_canons, idx_court_statute, idx_court_statute_count, doc_meta, CONFIG["budget_court_statute"])
ch_concept    = channel_concept(expanded_weighted, idx_concept_en, CONFIG["budget_concept"])
ch_term       = channel_term(targets, idx_term_orig, idx_term_lemma,
                              CONFIG["budget_term"], corpus_keys=term_orig_keys)
ch_per_area   = channel_per_area_bedrock(targets.get("legal_area_keywords", []),
                                          statute_target_codes, per_area_canon_count,
                                          idx_law_direct, CONFIG["budget_per_area"])
ch_cocit      = channel_co_citation(targets, co_neighbours, idx_law_direct, idx_court_statute,
                                     CONFIG["budget_co_citation"])

# BM25 (multi-language: per-language FTS5 indices, merged by max-score).
ch_bm25 = bm25_search_multilang(QUERY, targets, CONFIG["budget_bm25"])

# Diagnostic alias for Phase 11 token-overlap analysis (kept for back-compat
# after BM25 went multi-language). Concatenates query + all term/concept
# targets across languages — used only for query_tokens computation, not for
# the actual BM25 retrieval which now uses bm25_search_multilang.
bm25_query_text = QUERY + " " + " ".join(
    (targets.get("term_targets_de") or [])
  + (targets.get("term_targets_fr") or [])
  + (targets.get("concept_targets_en") or [])
)

# Vector (skipped if EMB_AVAILABLE = False)
ch_vector  = channel_vector(q_emb_raw,      CONFIG["budget_vector"])           if VECTOR_OK else []
ch_venrich = channel_vector(q_emb_enriched, CONFIG["budget_vector_enriched"])  if VECTOR_OK else []

# Build seed pool from topical channels for sibling + graph + backprop
def _court_hits(hits): return {d for d, _ in hits if doc_meta.get(d, {}).get("family") == "court"}
seed = (
    _court_hits(ch_court_stat) | _court_hits(ch_law_direct)
  | _court_hits(ch_concept)    | _court_hits(ch_term)
  | _court_hits(ch_per_area)   | _court_hits(ch_cocit)
  | _court_hits(ch_bm25)
  | _court_hits(ch_vector)     | _court_hits(ch_venrich)
)
print(f"seed pool: {len(seed):,} court doc_ids")

# Sibling (court_base — own-judgment) + graph (cross-judgment + case-level fan-out)
ch_sibling = channel_sibling(seed, idx_court_base, idx_judgment_importance, doc_meta, CONFIG["budget_sibling"])
print(f"sibling_expansion: {len(ch_sibling):,} expansions")

if GRAPH_OK:
    ch_graph_fwd = channel_graph_forward(seed, idx_graph_out, idx_judgment_importance, doc_meta, CONFIG["budget_graph_forward"])
    ch_graph_rev = channel_graph_reverse(seed, idx_graph_in, idx_judgment_importance, doc_meta, CONFIG["budget_graph_reverse"])
    print(f"graph_forward: {len(ch_graph_fwd):,} expansions")
    print(f"graph_reverse: {len(ch_graph_rev):,} expansions")
    if CONFIG["enable_graph_2hop"]:
        ch_graph_2h = channel_graph_2hop(seed, idx_graph_out, CONFIG["budget_graph_2hop"])
        print(f"graph_2hop:    {len(ch_graph_2h):,} expansions")
    else:
        ch_graph_2h = []
else:
    ch_graph_fwd = []; ch_graph_rev = []; ch_graph_2h = []

# statute_backprop (kept; uses row enrichment, complementary to graph)
backprop_seed = seed | _court_hits(ch_sibling) | _court_hits(ch_graph_fwd) | _court_hits(ch_graph_rev)
ch_backprop = channel_statute_backprop(
    backprop_seed, doc_statute_anchors, idx_law_direct, CONFIG["budget_backprop"]
)
print(f"backprop seed: {len(backprop_seed):,} -> {len(ch_backprop):,} law expansions")

CHANNELS = [
    ("law_direct_match",  ch_law_direct),
    ("court_statute",     ch_court_stat),
    ("co_citation",       ch_cocit),
    ("per_area_bedrock",  ch_per_area),
    ("statute_backprop",  ch_backprop),
    ("sibling_expansion", ch_sibling),
    ("graph_forward",     ch_graph_fwd),
    ("graph_reverse",     ch_graph_rev),
    ("graph_2hop",        ch_graph_2h),
    ("concept_en",        ch_concept),
    ("term_orig",         ch_term),
    ("bm25",              ch_bm25),
    ("vector_raw",        ch_vector),
    ("vector_enriched",   ch_venrich),
]

print()
print(f"{'channel':<22}  {'size':>6}  {'gold_in_ch':>11}  recall")
print("-" * 60)
for name, hits in CHANNELS:
    found = {d for d, _ in hits}
    g = len(found & gold_doc_set)
    print(f"{name:<22}  {len(hits):>6}  {g:>11}  {100*g/total_gold:5.1f}%")

union_did = set()
for _, hits in CHANNELS:
    union_did.update(d for d, _ in hits)
print()
print(f"Union: {len(union_did):,} unique doc_ids")
print(f"Gold in union: {len(union_did & gold_doc_set)}/{total_gold}  (UPPER BOUND on R@K)")

Statute canons: LLM=5, +co-citation=68
LLM target codes: {'StPO', 'BGG'}
Concepts: LLM=10 -> expanded=246
seed pool: 3,533 court doc_ids
sibling_expansion: 2,000 expansions
graph_forward: 1,500 expansions
graph_reverse: 1,000 expansions
backprop seed: 7,802 -> 800 law expansions

channel                   size   gold_in_ch  recall
------------------------------------------------------------
law_direct_match           152            7   16.7%
court_statute              600            0    0.0%
co_citation               1000            0    0.0%
per_area_bedrock           150            4    9.5%
statute_backprop           800           10   23.8%
sibling_expansion         2000            6   14.3%
graph_forward             1500           11   26.2%
graph_reverse             1000            0    0.0%
graph_2hop                   0            0    0.0%
concept_en                 600            9   21.4%
term_orig                  500            4    9.5%
bm25                       600    

# Phase 9 — RRF fusion + negative gate + final pool

## 9.1 What this cell does

- **RRF (Reciprocal Rank Fusion)**: each doc's fused score = sum over channels
  of `1 / (rrf_k + rank)`. Multi-channel hits bubble up; single-channel hits
  rank by their own rank.
- **Negative gate**: drop docs with `paragraph_role` in noise set or
  `is_notification_paragraph=True` (procedural boilerplate that rarely
  contains substantive law).
- **Guarantee channels**: docs in `law_direct_match`, `per_area_bedrock`,
  `statute_backprop` are PREPENDED to the final list. They get RRF-tail
  protection — even if their RRF score isn't great, they're force-included.
- **Final pool**: top `topk_final` (1000) after gating + guarantee.

## 9.2 Why this design

- RRF is robust to channel-score scale differences (BM25 negative-log,
  vector cosine [-1,1], counter scores arbitrary).
- Guarantees protect the highest-precision channels from being drowned by
  high-volume noise channels.
- Negative gate trims procedural-only paragraphs that have no legal
  substance — these never carry gold per `paragraph_role` enrichment.

## 9.3 Expected outputs

- Pre-gate fused: ~5000-7000 unique docs
- Post-gate: drop ~5%
- Guarantee pool: ~600 docs (LDM 68 + PAB 150 + SBP 400 within budgets)
- R@1000: target ≥ 0.60 (≥ 26/42)
- Stretch R@1000: ≥ 0.90 (≥ 38/42)

## 9.4 Failure modes

- R@1000 < 0.50 → fusion isn't lifting graph_forward hits. Check Phase 8
  showed graph_forward catching gold; if yes, RRF rank is too low (single-
  channel score 0.0167 max). Consider promoting graph_forward to guarantee.
- R@1000 < 0.20 → catastrophic: a major channel is empty. Check Phase 8
  per-channel sizes.

In [ ]:
def rrf_fuse(channels, k, weights=None):
    """v7.4 weighted RRF: score(did) = sum_ch weights[ch] / (k + rank + 1)."""
    if weights is None: weights = {}
    score = defaultdict(float)
    for name, hits in channels:
        w = weights.get(name, 1.0)
        if w == 0: continue
        for rank, (did, _) in enumerate(hits):
            score[did] += w / (k + rank + 1)
    return score

# v7.4: substantive paragraph_role overrides buggy is_notification_paragraph.
SUBSTANTIVE_ROLES = {
    "facts", "reasoning", "legal_standard", "application",
    "holding", "citation", "procedural_history",
}

def apply_neg_gate(doc_ids, doc_meta, noise_roles):
    keep = []
    for did in doc_ids:
        m = doc_meta.get(did) or {}
        pr = (m.get("paragraph_role") or "").lower()
        if pr in SUBSTANTIVE_ROLES:
            keep.append(did); continue
        if m.get("is_notification_paragraph"): continue
        if pr in noise_roles: continue
        keep.append(did)
    return keep

def round_robin_guarantee(channels_by_name, guarantee_channel_names,
                           per_channel_cap, total_cap):
    """v7.4 round-robin: each channel contributes 1 per round up to
    per_channel_cap, until total_cap reached."""
    iters = {cn: iter(channels_by_name.get(cn, [])) for cn in guarantee_channel_names}
    counts = {cn: 0 for cn in guarantee_channel_names}
    out = []; seen = set()
    while iters and len(out) < total_cap:
        exhausted = []
        for cn in list(iters.keys()):
            if counts[cn] >= per_channel_cap:
                exhausted.append(cn); continue
            try:
                did, _ = next(iters[cn])
                while did in seen:
                    did, _ = next(iters[cn])
                out.append(did); seen.add(did); counts[cn] += 1
                if len(out) >= total_cap: break
            except StopIteration:
                exhausted.append(cn)
        for cn in exhausted:
            if cn in iters: del iters[cn]
    return out

rrf_scores = rrf_fuse(CHANNELS, CONFIG["rrf_k"], weights=CONFIG.get("channel_weights"))
ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
ranked_dids = [d for d, _ in ranked]
ranked_gated = apply_neg_gate(ranked_dids, doc_meta, CONFIG["noise_paragraph_roles"])

ch_by_name = dict(CHANNELS)
guarantee = round_robin_guarantee(
    ch_by_name, CONFIG["guarantee_channels"],
    CONFIG.get("guarantee_per_channel", 400),
    CONFIG["topk_final"],
)
guarantee = apply_neg_gate(guarantee, doc_meta, CONFIG["noise_paragraph_roles"])

PASS_K = CONFIG["topk_final"]
final_topk = list(guarantee); seen_f = set(final_topk)
for did in ranked_gated:
    if len(final_topk) >= PASS_K: break
    if did not in seen_f:
        final_topk.append(did); seen_f.add(did)
final_topk = final_topk[:PASS_K]

print(f"Pre-gate fused: {len(ranked_dids):,}")
print(f"Post-gate:      {len(ranked_gated):,}")
print(f"Guarantee pool: {len(guarantee):,}  (channels: {CONFIG['guarantee_channels']})")
print(f"Final top-{PASS_K}: {len(final_topk):,}")
print()
print("R@K curve:")
print(f"{'K':>6}  gold/{total_gold}  recall")
print("-" * 30)
all_ranked = guarantee + [d for d in ranked_gated if d not in set(guarantee)]
for K in [50, 100, 200, 300, 500, 750, 1000, 1500, 2000, 3000, 5000]:
    K_use = min(K, len(all_ranked))
    g = len(set(all_ranked[:K_use]) & gold_doc_set)
    print(f"{K_use:>6}  {g:>5}/{total_gold}  {100*g/total_gold:5.1f}%")
    if K_use == len(all_ranked): break

R_at_K = len(set(final_topk) & gold_doc_set) / max(1, total_gold)
print()
print(f"Final R@{PASS_K} = {R_at_K:.3f}  ({len(set(final_topk) & gold_doc_set)}/{total_gold})")


Pre-gate fused: 8,958
Post-gate:      8,770
Guarantee pool: 848  (channels: ['law_direct_match', 'per_area_bedrock', 'statute_backprop'])
Final top-1000: 1,000

R@K curve:
     K  gold/42  recall
------------------------------
    50      3/42    7.1%
   100      5/42   11.9%
   200      7/42   16.7%
   300      7/42   16.7%
   500      8/42   19.0%
   750     10/42   23.8%
  1000     17/42   40.5%
  1500     21/42   50.0%
  2000     26/42   61.9%
  3000     27/42   64.3%
  5000     30/42   71.4%

Final R@1000 = 0.405  (17/42)


# Phase 10 — DIAGNOSIS (per-gold root-cause analysis)

This is the centerpiece of v7. For every gold citation NOT in top-1000,
we run a **per-channel signal trace** and print a structured report:

1. **Source row inspection** — is this gold present as a `citation` column
   value in laws_de or court_considerations? If yes, we print:
   - The full `search_text` (BM25 input)
   - The enrichment fields (concepts_en, terms, statute_anchors)
   - The doc's paragraph_role, court_base, legal_area_static

2. **Per-channel hit/miss + reason**:
   - For each of the 14 channels, report:
     - `hit (rank N)` — gold is in channel's output at rank N
     - `miss (no signal)` — gold's doc_id never appeared as a candidate
     - `miss (truncated)` — gold appeared in raw scoring but got cut by budget
     - `miss (zero overlap)` — for token channels, no token-level overlap
   - For **bm25**: which query tokens (if any) appear in gold's `search_text`?
   - For **vector**: cosine similarity between query embedding and gold's
     row embedding (requires gold to be in the manifest).
   - For **concept_en / term_orig**: which expanded LLM concepts/terms map
     to this gold? If none, it's a coverage gap in query expansion.
   - For **graph_forward**: from each seed, list outgoing edges that target
     this gold. If empty, no caught seed text-cites this gold.
   - For **graph_reverse**: list incoming edges from this gold. If a source
     in the seed set, why wasn't its edge followed? (Should never happen.)
   - For **sibling_expansion**: gold's `court_base`. List other Es of the
     same judgment. Were any caught? If none caught, sibling can't fire.

3. **Recommended fix** — one of:
   - Enrichment gap: gold's `search_text` doesn't contain query keywords.
     Fix: extend law/court LLM enrichment prompt.
   - Query expansion gap: LLM didn't produce a relevant concept/term that
     matches this gold's enrichment. Fix: improve query-expansion prompt
     or use a stronger model.
   - Graph extraction gap: gold's referencing source row exists but no
     edge to it in the graph. Fix: add a regex pattern or run another
     extraction pass.
   - RRF rank gap: gold has hits in N channels but score is below cutoff.
     Fix: lift channel budget OR add channel to guarantee_channels.
   - True orphan: nothing in the corpus references this gold by exact
     pinpoint. Fix: case-level fan-out (already in graph) — verify it fired.

## 10.1 Helper functions for diagnosis

In [ ]:
# --- per-channel rank lookup helpers -----------------------------------------
def channel_rank(did, hits):
    """Return rank (0-indexed) of did in hits, or None if absent."""
    for i, (d, _) in enumerate(hits):
        if d == did: return i
    return None

def channel_score(did, hits):
    for d, s in hits:
        if d == did: return s
    return None

# --- token overlap for BM25 --------------------------------------------------
def _tokens(text):
    return set(t for t in re.split(r"[^\w\d]+", (text or "").lower(), flags=re.UNICODE)
               if len(t) >= CONFIG["bm25_min_token_len"])

query_tokens = _tokens(bm25_query_text)
print(f"Query token count (post-enhance): {len(query_tokens)}")

# --- vector cosine helper (requires manifest mapping) ------------------------
def gold_vector_cos(did, q_emb):
    if not VECTOR_OK: return None
    if did not in row_for_did: return None
    ridx = row_for_did[did]
    with torch.no_grad():
        v = E_GPU[ridx]
        v = v / (v.norm() + 1e-9)
        q = q_emb.to(E_GPU.device, dtype=E_GPU.dtype)
        q = q / (q.norm() + 1e-9)
        return float((v * q).sum().item())

import torch
# Compute gold vector cos for all gold (those in manifest)
gold_vec_cos_raw = {}
gold_vec_cos_enr = {}
if VECTOR_OK:
    for did in gold_doc_set:
        cr = gold_vector_cos(did, q_emb_raw)
        ce = gold_vector_cos(did, q_emb_enriched)
        if cr is not None: gold_vec_cos_raw[did] = cr
        if ce is not None: gold_vec_cos_enr[did] = ce

# --- which expanded concepts/terms point to a given did ----------------------
def concepts_pointing_to(did, expanded_concepts, idx):
    return [c for c in expanded_concepts if did in idx.get(c, ())]
def terms_pointing_to(did, targets, idx):
    out = []
    for key in ("term_targets_de", "term_targets_fr"):
        for raw in targets.get(key, []) or []:
            tok = norm_token(raw, CONFIG["lowercase_terms"])
            if tok and did in idx.get(tok, ()):
                out.append((key, raw, tok))
    return out

# --- find caught seeds with edges to a given did -----------------------------
def graph_paths_to(did, seed, idx_graph_in):
    """Return list of seeds that have an outgoing edge to did (i.e., did is
    reachable forward from these seeds)."""
    incoming_to_did = set(idx_graph_in.get(did, ()))
    return [s for s in seed if s in incoming_to_did]

def graph_paths_from(did, seed, idx_graph_out):
    """Return list of seeds that have an incoming edge from did (i.e., this
    did's outgoing edges include some seed = caught)."""
    outgoing_from_did = set(idx_graph_out.get(did, ()))
    return [s for s in seed if s in outgoing_from_did]

# --- court_base sibling diagnosis --------------------------------------------
def sibling_diagnosis(did, doc_meta, idx_court_base, seed):
    m = doc_meta.get(did) or {}
    cb = m.get("court_base")
    if not cb: return None, [], []
    siblings = idx_court_base.get(cb, set()) - {did}
    caught_siblings = [s for s in siblings if s in seed]
    return cb, list(siblings), caught_siblings

Query token count (post-enhance): 122


## 10.2 Per-gold trace (the main diagnostic loop)

For each gold not in top-1000, prints a structured report. If a gold IS in
top-1000, we still record at what rank for the summary table.

In [ ]:
top1000_set = set(final_topk)
gold_in_top = gold_doc_set & top1000_set
gold_missed = gold_doc_set - top1000_set

# Reverse map: doc_id -> citation
did_to_cit = {did: m["citation"] for did, m in doc_meta.items()}

print(f"R@1000 = {len(gold_in_top)}/{total_gold} = {100*len(gold_in_top)/total_gold:.1f}%")
print(f"Missed gold: {len(gold_missed)} of {total_gold}")
print()
print("=" * 80)
print("PER-GOLD MISS DIAGNOSIS")
print("=" * 80)

miss_summary = []  # (gold_cit, root_cause, fix_suggestion)
for did in sorted(gold_missed):
    cit = did_to_cit.get(did, did)
    m = doc_meta.get(did, {})
    fam = m.get("family")
    cb = m.get("court_base")
    pr = m.get("paragraph_role")
    print()
    print("-" * 80)
    print(f"GOLD: {cit}")
    print(f"  doc_id={did}  family={fam}  court_base={cb}  paragraph_role={pr}")

    # --- Source row inspection ---
    st = search_text.get(did, "")
    print(f"  search_text ({len(st)} chars): {st[:240]}{'...' if len(st)>240 else ''}")

    # --- Per-channel rank ---
    print("  Per-channel:")
    channel_hit_count = 0
    channel_miss_reasons = []
    for cname, hits in CHANNELS:
        r = channel_rank(did, hits)
        if r is not None:
            print(f"    HIT  {cname:<22} rank={r}  (size={len(hits)})")
            channel_hit_count += 1
        else:
            # Probe why miss
            reason = None
            if cname == "bm25":
                doc_tokens = _tokens(st)
                overlap = query_tokens & doc_tokens
                reason = f"token overlap with query: {len(overlap)} ({sorted(overlap)[:6]})"
            elif cname == "vector_raw" and VECTOR_OK:
                cr = gold_vec_cos_raw.get(did)
                reason = f"cos_sim_raw={cr:.4f}" if cr is not None else "not in manifest"
            elif cname == "vector_enriched" and VECTOR_OK:
                ce = gold_vec_cos_enr.get(did)
                reason = f"cos_sim_enriched={ce:.4f}" if ce is not None else "not in manifest"
            elif cname == "concept_en":
                ptrs = concepts_pointing_to(did, expanded_concepts, idx_concept_en)
                reason = f"expanded concepts hitting this row: {ptrs[:5]}" if ptrs else "no concept hits this row"
            elif cname == "term_orig":
                tptrs = terms_pointing_to(did, targets, idx_term_orig)
                reason = f"terms hitting this row: {tptrs[:3]}" if tptrs else "no term hits this row"
            elif cname == "graph_forward":
                paths = graph_paths_to(did, seed, idx_graph_in)
                reason = f"caught seeds with edges -> {len(paths)}: {paths[:3]}"
            elif cname == "graph_reverse":
                paths = graph_paths_from(did, seed, idx_graph_out)
                reason = f"caught seeds reachable from this -> {len(paths)}: {paths[:3]}"
            elif cname == "sibling_expansion":
                cb_, sibs, caught_sibs = sibling_diagnosis(did, doc_meta, idx_court_base, seed)
                reason = f"court_base={cb_} | sibs={len(sibs)} | caught_sibs={len(caught_sibs)}"
            elif cname == "law_direct_match":
                canon = statute_anchor_canonical(cit)
                reason = f"canonical={canon} | in_canon_set={canon in co_expanded_canons}"
            elif cname == "court_statute":
                row_canons = doc_statute_anchors.get(did, set())
                hits_named = row_canons & llm_statute_canons
                reason = f"row_anchors={list(row_canons)[:3]} | overlap with LLM names={list(hits_named)}"
            elif cname == "statute_backprop":
                row_canons = doc_statute_anchors.get(did, set())
                # for law gold, count caught court rows whose anchors include this canon
                gcanon = statute_anchor_canonical(cit) if fam == "law" else None
                if gcanon:
                    n_courts = sum(1 for s in seed if gcanon in doc_statute_anchors.get(s, set()))
                    reason = f"law canonical={gcanon} | seed-courts-citing={n_courts}"
                else:
                    reason = f"family={fam}, statute_backprop only emits law rows"
            elif cname == "co_citation":
                gcanon = statute_anchor_canonical(cit) if fam == "law" else None
                ndid_in_neighbours = False
                for raw in targets.get("statute_targets", []) or []:
                    tcanon = statute_anchor_canonical(raw)
                    if not tcanon: continue
                    for nb, _ in co_neighbours.get(tcanon, []):
                        if did in idx_law_direct.get(nb, set()) or did in idx_court_statute.get(nb, set()):
                            ndid_in_neighbours = True; break
                    if ndid_in_neighbours: break
                reason = f"reachable via co-cited neighbour={ndid_in_neighbours}"
            elif cname == "per_area_bedrock":
                gcanon = statute_anchor_canonical(cit) if fam == "law" else None
                in_pab = False
                if gcanon:
                    for area, cnts in per_area_canon_count.items():
                        if any(k.lower() in area for k in (targets.get("legal_area_keywords") or [])):
                            if gcanon in dict(cnts.most_common(CONFIG["per_area_top_n"])):
                                in_pab = True; break
                reason = f"law canonical={gcanon} | in selected-area top-{CONFIG['per_area_top_n']}={in_pab}"
            elif cname == "graph_2hop":
                reason = "disabled by config" if not CONFIG["enable_graph_2hop"] else "miss"
            else:
                reason = "miss"
            print(f"    MISS {cname:<22} ({reason})")
            channel_miss_reasons.append((cname, reason))

    # --- RRF rank if any score ---
    rrf_s = rrf_scores.get(did, 0.0)
    rrf_pos = ranked_dids.index(did) if did in rrf_scores else None
    print(f"  RRF: score={rrf_s:.5f}  rank={rrf_pos}  (top-1000 cutoff is rank 999)")

    # --- root cause heuristic ---
    if channel_hit_count == 0:
        cause = "NO_CHANNEL_HIT"
        fix = "no channel produced this as a candidate; check enrichment, regex, graph"
    elif rrf_s == 0:
        cause = "RRF_ZERO"
        fix = "channel sizes vs guarantees mismatch (should not happen)"
    elif rrf_pos is not None and rrf_pos < 1000 and did not in top1000_set:
        cause = "GATED_OUT"
        fix = f"negative gate dropped this; paragraph_role={pr}"
    elif rrf_pos is not None and rrf_pos >= 1000:
        cause = "RRF_RANK_TOO_LOW"
        fix = f"in {channel_hit_count} channel(s) but RRF rank {rrf_pos}; consider promoting a channel to guarantee or lifting budget"
    else:
        cause = "UNKNOWN"
        fix = "investigate above trace"

    print(f"  ROOT CAUSE: {cause}")
    print(f"  FIX SUGGESTION: {fix}")
    miss_summary.append((cit, cause, channel_hit_count, fix))

R@1000 = 17/42 = 40.5%
Missed gold: 25 of 42

PER-GOLD MISS DIAGNOSIS

--------------------------------------------------------------------------------
GOLD: 1B_210/2023 E. 4.1
  doc_id=court:1052533  family=court  court_base=1B_210/2023  paragraph_role=legal_standard
  search_text (1466 chars): 1B_210/2023 E. 4.1 4.1. Conformément à l'art. 221 al. 1 let. b CPP, la détention provisoire ou pour motifs de sûreté ne peut être ordonnée que lorsque le prévenu est fortement soupçonné d'avoir commis un crime ou un délit et qu'il y a série...
  Per-channel:
    MISS law_direct_match       (canonical=None | in_canon_set=False)
    MISS court_statute          (row_anchors=['221 StPO'] | overlap with LLM names=['221 StPO'])
    MISS co_citation            (reachable via co-cited neighbour=False)
    MISS per_area_bedrock       (law canonical=None | in selected-area top-100=False)
    MISS statute_backprop       (family=court, statute_backprop only emits law rows)
    MISS sibling_expansion      (

## 10.3 Failure-mode summary (aggregated)

Aggregate the per-gold root causes into buckets so we know where to invest
the next iteration's effort.

In [ ]:
print("=" * 80)
print("FAILURE-MODE SUMMARY")
print("=" * 80)

from collections import Counter as _Counter
cause_counter = _Counter(c for _, c, _, _ in miss_summary)
print(f"Missed gold: {len(miss_summary)}")
print(f"  by root cause:")
for cause, n in cause_counter.most_common():
    print(f"    {cause:<22}  {n}")

# Top fixes
print()
print("Top 10 missed gold with their fix suggestions:")
for cit, cause, n_hits, fix in miss_summary[:10]:
    print(f"  - {cit:<35}  cause={cause:<22}  channels_hit={n_hits}  fix={fix}")

# Quick yes/no diagnostic table
print()
print("=" * 80)
print("CHANNEL-LEVEL HIT RATE ON MISSED GOLD")
print("=" * 80)
print("(a channel that 'caught' missed gold means the gold was in its candidate")
print(" pool but ranked below top-1000 after fusion)")
print()
print(f"{'channel':<22}  caught_missed_gold")
print("-" * 50)
for cname, hits in CHANNELS:
    hit_set = {d for d, _ in hits}
    n_caught = len(hit_set & gold_missed)
    print(f"  {cname:<20}  {n_caught}")

FAILURE-MODE SUMMARY
Missed gold: 25
  by root cause:
    NO_CHANNEL_HIT          12
    RRF_RANK_TOO_LOW        9
    GATED_OUT               4

Top 10 missed gold with their fix suggestions:
  - 1B_210/2023 E. 4.1                   cause=RRF_RANK_TOO_LOW        channels_hit=1  fix=in 1 channel(s) but RRF rank 2993; consider promoting a channel to guarantee or lifting budget
  - 1B_536/2018 E. 5.1                   cause=RRF_RANK_TOO_LOW        channels_hit=1  fix=in 1 channel(s) but RRF rank 3799; consider promoting a channel to guarantee or lifting budget
  - 7B_496/2025 E. 3.2                   cause=NO_CHANNEL_HIT          channels_hit=0  fix=no channel produced this as a candidate; check enrichment, regex, graph
  - BGE 133 I 168 E. 4.1                 cause=RRF_RANK_TOO_LOW        channels_hit=1  fix=in 1 channel(s) but RRF rank 4839; consider promoting a channel to guarantee or lifting budget
  - 1B_90/2021 E. 2.4                    cause=RRF_RANK_TOO_LOW        channels_hit=1 

# Phase 11 — Save artifacts + cleanup

## 11.1 What we save

- `final_topk.json` — list of (citation, doc_id, rank) for top-1000.
- `gold_in_top.json` — gold citations captured.
- `gold_missed_diagnosis.json` — per-gold root-cause + fix suggestion.
- `targets.json` — LLM query expansion output (for repro).
- `config.json` — exact CONFIG used.
- `summary.json` — R@K numbers + channel sizes.

These persist in `out_dir` so a future run can compare.

In [ ]:
import json as _json
out_dir = PATHS["out_dir"]
out_dir.mkdir(parents=True, exist_ok=True)

# Final top-K
final_top_records = [
    {"rank": i, "doc_id": did, "citation": did_to_cit.get(did, did)}
    for i, did in enumerate(final_topk)
]
(out_dir / "final_topk.json").write_text(_json.dumps(final_top_records, ensure_ascii=False, indent=2), encoding="utf-8")

# Gold captured + missed
gold_top_records = sorted(
    [{"doc_id": d, "citation": did_to_cit.get(d, d), "rank": final_topk.index(d) if d in set(final_topk) else None}
     for d in gold_in_top],
    key=lambda x: x["rank"] if x["rank"] is not None else 1e9,
)
(out_dir / "gold_in_top.json").write_text(_json.dumps(gold_top_records, ensure_ascii=False, indent=2), encoding="utf-8")

miss_records = [
    {"citation": cit, "root_cause": cause, "channels_hit": n_hits, "fix_suggestion": fix}
    for cit, cause, n_hits, fix in miss_summary
]
(out_dir / "gold_missed_diagnosis.json").write_text(_json.dumps(miss_records, ensure_ascii=False, indent=2), encoding="utf-8")

# LLM targets + config
(out_dir / "targets.json").write_text(_json.dumps(targets, ensure_ascii=False, indent=2), encoding="utf-8")
(out_dir / "config.json").write_text(_json.dumps(
    {k: v for k, v in CONFIG.items() if not isinstance(v, set)},
    ensure_ascii=False, indent=2, default=str), encoding="utf-8")

# Summary
summary = {
    "query_id": "val_001",
    "total_gold": total_gold,
    "gold_in_corpus": len(gold_doc_set),
    "R_at_1000": len(gold_in_top) / max(1, total_gold),
    "channel_sizes": {n: len(h) for n, h in CHANNELS},
    "channel_recall": {n: len({d for d, _ in h} & gold_doc_set) for n, h in CHANNELS},
    "union_upper_bound": len(union_did & gold_doc_set) / max(1, total_gold),
}
(out_dir / "summary.json").write_text(_json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Saved 6 artifacts to {out_dir}")

Saved 6 artifacts to /content/drive/MyDrive/swiss_law/research/anchor_funnel_val001_v7


## 11.2 Cleanup (free GPU memory)

**What:** Delete large GPU tensors so a notebook re-run starts clean.

**Why:** Colab keeps state across cells; without explicit cleanup, re-running
Phase 6 will OOM.

In [ ]:
import gc
try:
    del E_GPU
except NameError:
    pass
try:
    del EMB_MODEL
except NameError:
    pass
gc.collect()
import torch as _torch2
if _torch2.cuda.is_available():
    _torch2.cuda.empty_cache()
    print(f"VRAM after cleanup: {_torch2.cuda.memory_allocated()/1024**3:.2f} GB")
print("cleaned up.")

VRAM after cleanup: 0.01 GB
cleaned up.
